# M4 — Road Damage Object Detection (YOLO11n)

**Submission-ready organization of the uploaded working notebook**

This notebook preserves the original code of the retained analytical cells, in their original order, and removes repetitive troubleshooting, notebook self-editing, and duplicate setup operations. The original upload is not changed.

**Data protocol:** Train: 23,016 images; validation: 7,207 images; independent Norway test: 8,161 images. The four detection classes are D00, D10, D20, D40. Norway was not used for training or validation.

**Reported final Norway performance:** Precision **0.2410**, recall **0.1235**, mAP@50 **0.0792**, mAP@50–95 **0.0319** (11,229 annotated instances). These numbers are transcribed from the results shared in the conversation, **not recomputed from this uploaded notebook**.

**Reproducibility notes:** The notebook refers to Google Drive data and checkpoints that are **not included** in the `.ipynb` upload. Local Colab data under `/content/m4_local` is temporary. The uploaded notebook contains no saved cell outputs; opening this notebook alone will not display the original figures or evaluation logs. Avoid **Run all**: preparation, copying, training, and final test cells are intentionally retained as provenance.

**Limitations:** Norway domain shift and class imbalance; limited D40 detection; some images lack supported damage annotations and should not automatically be interpreted as verified negatives; absolute Drive paths require adaptation in other environments.


## 1. Environment and project paths

Mount Google Drive and locate the existing project. The Drive folder `AI Project ` includes a trailing space.

In [ ]:

from google.colab import drive
from pathlib import Path

# Connect Google Drive
drive.mount('/content/drive')

# Locate Google Drive folders
root = Path('/content/drive/MyDrive')

print("\nGoogle Drive folders:")
for folder in sorted(root.iterdir()):
    if folder.is_dir():
        print(" -", folder.name)


In [ ]:

from pathlib import Path

# Locate the folder automatically
root = Path("/content/drive/MyDrive")

project_dir = next(
    p for p in root.iterdir()
    if p.is_dir() and p.name.strip() == "AI Project"
)

print("Project folder:", repr(project_dir.name))
print("\nContents:\n")

for item in sorted(project_dir.iterdir()):
    if item.is_dir():
        print("📁 FOLDER:", item.name)
    else:
        print("📄 FILE:", item.name)


## 2. M1 manifests and data integrity

Read M1 metadata without modifying its splits or annotations. Norway remains the independent test-country holdout.

In [ ]:

import pandas as pd

# Read existing M1 files without modifying them
splits = pd.read_csv(data_dir / "split_manifest.csv")
annotations = pd.read_csv(data_dir / "annotations.csv")

print("=== SPLIT MANIFEST ===")
print("Total images:", len(splits))
print("Columns:", splits.columns.tolist())

print("\nImages per split:")
print(splits["split"].value_counts())

print("\nImages per country and split:")
print(pd.crosstab(splits["country"], splits["split"]))

print("\n=== ANNOTATIONS ===")
print("Total annotations:", len(annotations))
print("Columns:", annotations.columns.tolist())

print("\nAnnotations per class:")
print(annotations["class"].value_counts())

print("\nFirst 3 annotations:")
print(annotations.head(3).to_string(index=False))


In [ ]:

from zipfile import ZipFile
from collections import Counter
from pathlib import PureWindowsPath

print("=== M4 IMAGE MATCHING CHECK ===\n")

# Collect image filenames from all ZIP archives
archive_images = {}

for zip_path in sorted(zip_dir.glob("*.zip")):
    with ZipFile(zip_path) as z:
        archive_images[zip_path.stem] = {
            PureWindowsPath(name).name
            for name in z.namelist()
            if name.lower().endswith(
                (".jpg", ".jpeg", ".png")
            )
        }

# Match M1 country names to ZIP names
country_map = {
    "China_Drone": "China_Drone",
    "China_MotorBike": "China_MotorBike",
    "Czech": "Czech",
    "India": "India",
    "Japan": "Japan",
    "Norway-002": "Norway",
    "United_States": "United_States"
}

missing = []
matched = 0

for row in splits.itertuples(index=False):
    archive = country_map[row.country]
    filename = PureWindowsPath(row.image_path).name

    if filename in archive_images[archive]:
        matched += 1
    else:
        missing.append((row.image_id, archive, filename))

print("M1 images:", len(splits))
print("Matched in ZIP archives:", matched)
print("Missing from ZIP archives:", len(missing))

if missing:
    print("\nFirst 10 missing:")
    for item in missing[:10]:
        print(item)

print("\nSplit distribution:")
print(splits["split"].value_counts())


In [ ]:

import pandas as pd

# Match annotations to M1 image metadata
check = annotations.merge(
    splits[["image_id", "width", "height", "split"]],
    on="image_id",
    how="left",
    suffixes=("_annotation", "_manifest"),
    validate="many_to_one",
    indicator=True
)

print("=== M4 ANNOTATION VERIFICATION ===")

print("Total annotations:", len(check))

# Check annotation-to-image matching
unmatched = (check["_merge"] != "both").sum()
print("Annotations without M1 image:", unmatched)

# Check that split assignments agree
split_mismatch = (
    check["split_annotation"] != check["split_manifest"]
).sum()

print("Split mismatches:", split_mismatch)

# Check bounding boxes against image dimensions
valid_bounds = (
    (check["xmin"] >= 0) &
    (check["ymin"] >= 0) &
    (check["xmax"] > check["xmin"]) &
    (check["ymax"] > check["ymin"]) &
    (check["xmax"] <= check["width"]) &
    (check["ymax"] <= check["height"])
)

print("Valid bounding boxes:", int(valid_bounds.sum()))
print("Invalid bounding boxes:", int((~valid_bounds).sum()))

# Check all expected classes
expected_classes = {"D00", "D10", "D20", "D40"}
found_classes = set(annotations["class"].dropna())

print("Unexpected classes:", found_classes - expected_classes)

print("\nAnnotations by split:")
print(annotations["split"].value_counts())


In [ ]:

import pandas as pd

audit_path = data_dir / "m1_visual_label_audit.csv"
audit = pd.read_csv(audit_path)

print("=== M1 LABEL AUDIT ===")
print("Total reviewed records:", len(audit))
print("\nColumns:")
print(audit.columns.tolist())

print("\nAudit results:")
for col in audit.columns:
    if any(word in col.lower() for word in
           ["status", "decision", "result", "correct"]):
        print(f"\n{col}:")
        print(audit[col].value_counts(dropna=False))

print("\nFirst 5 records:")
print(audit.head(5).to_string(index=False))


In [ ]:

from pathlib import Path

# Create a separate M4 workspace
m4_dir = project_dir / "M4_Object_Detection"

folders = [
    "data",
    "configs",
    "notebooks",
    "models",
    "results",
    "reports"
]

m4_dir.mkdir(parents=True, exist_ok=True)

for folder in folders:
    (m4_dir / folder).mkdir(exist_ok=True)

print("=== M4 WORKSPACE CREATED ===")
print("Location:", m4_dir)

for folder in folders:
    path = m4_dir / folder
    print(folder, ":", path.exists())


In [ ]:

from zipfile import ZipFile
from pathlib import PureWindowsPath
from collections import Counter

print("=== M4 EXACT IMAGE MAPPING CHECK ===")

country_map = {
    "China_Drone": "China_Drone",
    "China_MotorBike": "China_MotorBike",
    "Czech": "Czech",
    "India": "India",
    "Japan": "Japan",
    "Norway-002": "Norway",
    "United_States": "United_States"
}

missing = []
ambiguous = []
matched = 0

for country, archive_name in country_map.items():

    zip_path = zip_dir / f"{archive_name}.zip"

    with ZipFile(zip_path) as z:
        image_index = {}

        for name in z.namelist():
            if name.lower().endswith(
                (".jpg", ".jpeg", ".png")
            ):
                filename = name.rsplit("/", 1)[-1]
                image_index.setdefault(filename, []).append(name)

        country_rows = splits[splits["country"] == country]

        for row in country_rows.itertuples(index=False):
            filename = PureWindowsPath(row.image_path).name
            matches = image_index.get(filename, [])

            if len(matches) == 1:
                matched += 1
            elif len(matches) == 0:
                missing.append(row.image_id)
            else:
                ambiguous.append((row.image_id, matches))

print("Total M1 images:", len(splits))
print("Unique matches:", matched)
print("Missing:", len(missing))
print("Ambiguous matches:", len(ambiguous))

if missing:
    print("\nFirst missing:", missing[:5])

if ambiguous:
    print("\nFirst ambiguous:", ambiguous[:5])


In [ ]:

import pandas as pd

# Original M1 data folder (read-only)
M1_DATA = project_dir / "Data"

split_file = M1_DATA / "split_manifest.csv"
annotations_file = M1_DATA / "annotations.csv"

print("=== M4 SOURCE FILE VERIFICATION ===")

print("Split manifest exists:", split_file.is_file())
print("Annotations exist:", annotations_file.is_file())

if split_file.is_file() and annotations_file.is_file():

    splits = pd.read_csv(split_file)
    annotations = pd.read_csv(annotations_file)

    print("\nSplit manifest rows:", len(splits))
    print("Annotation rows:", len(annotations))

    print("\nSplit columns:")
    print(splits.columns.tolist())

    print("\nAnnotation columns:")
    print(annotations.columns.tolist())

    print("\nDamage classes:")
    print(annotations["class"].value_counts())


In [ ]:

from pathlib import Path
import pandas as pd

# Read existing M1 manifests without modifying them
split_df = pd.read_csv(data_dir / "split_manifest.csv")
ann_df = pd.read_csv(data_dir / "annotations.csv")

print("=== M4 DATASET PREPARATION CHECK ===")

print("\nM1 split counts:")
print(split_df["split"].value_counts())

print("\nAnnotation counts:")
print(ann_df["split"].value_counts())

print("\nCountry distribution:")
print(pd.crosstab(
    split_df["country"],
    split_df["split"]
))

print("\nDuplicate image IDs:")
print(split_df["image_id"].duplicated().sum())

print("\nMissing annotation image references:")
print(
    (~ann_df["image_id"].isin(
        split_df["image_id"]
    )).sum()
)

print("\nM4 workspace:")
print(project_dir / "M4_Object_Detection")

print("\nOriginal M1 files remain unchanged.")


In [ ]:

import numpy as np

print("=== M4 BOUNDING BOX DIMENSION CHECK ===")

# Join annotations with image dimensions from M1
boxes = annotations.merge(
    splits[["image_id", "width", "height"]],
    on="image_id",
    how="left",
    validate="many_to_one"
)

# Check missing dimensions
missing_dimensions = (
    boxes["width"].isna() |
    boxes["height"].isna()
)

# Check coordinates against image boundaries
invalid_boxes = (
    (boxes["xmin"] < 0) |
    (boxes["ymin"] < 0) |
    (boxes["xmax"] > boxes["width"]) |
    (boxes["ymax"] > boxes["height"]) |
    (boxes["xmin"] >= boxes["xmax"]) |
    (boxes["ymin"] >= boxes["ymax"]) |
    (boxes["width"] <= 0) |
    (boxes["height"] <= 0)
)

print("Total annotations:", len(boxes))
print("Missing image dimensions:", missing_dimensions.sum())
print("Invalid/out-of-bound boxes:", invalid_boxes.sum())

print("\n=== RESULTS ===")

if missing_dimensions.sum() == 0 and invalid_boxes.sum() == 0:
    print("PASS: All bounding boxes fit within image dimensions.")
else:
    print("REVIEW REQUIRED: Some bounding boxes need inspection.")


## 3. YOLO label preparation

Generate four-class YOLO labels using the existing M1 split assignments. This is dataset preparation, not model training.

In [ ]:

from pathlib import Path

# Existing M4 workspace
m4_dir = project_dir / "M4_Object_Detection"

# New M4-only YOLO dataset directory
yolo_dir = m4_dir / "data" / "yolo_dataset"

# Create separate image and label folders
for split in ["train", "val", "test"]:
    for folder in ["images", "labels"]:
        (yolo_dir / folder / split).mkdir(
            parents=True,
            exist_ok=True
        )

print("=== M4 YOLO DATASET STRUCTURE ===")
print("YOLO directory:", yolo_dir)

for split in ["train", "val", "test"]:
    print(f"\n{split.upper()}")
    print("Images:", (yolo_dir / "images" / split).exists())
    print("Labels:", (yolo_dir / "labels" / split).exists())

print("\nOriginal M1 files remain unchanged.")


In [ ]:

# M4 — Step 2: Verify YOLO class mapping
# Read-only verification; no files are modified

class_map = {
    "D00": 0,
    "D10": 1,
    "D20": 2,
    "D40": 3
}

print("=== M4 YOLO CLASS MAPPING ===")

for class_name, class_id in class_map.items():
    print(f"{class_name} -> {class_id}")

# Use the existing M1 annotations DataFrame
observed_classes = set(annotations["class"].dropna().unique())
expected_classes = set(class_map.keys())

print("\nObserved M1 classes:", sorted(observed_classes))
print("Unexpected classes:", observed_classes - expected_classes)
print("Missing expected classes:", expected_classes - observed_classes)

assert observed_classes == expected_classes, \
    "Class mapping mismatch — stop before conversion."

print("\nPASS: M1 classes match the YOLO mapping.")


In [ ]:
import pandas as pd

# Load the existing M1 split manifest from Google Drive
split_manifest = pd.read_csv(
    data_dir / "split_manifest.csv"
)

print("=== M1 SPLIT MANIFEST LOADED ===")
print("Total images:", len(split_manifest))
print("\nSplit distribution:")
print(split_manifest["split"].value_counts())

print("\nPASS: split_manifest is ready.")

In [ ]:

# M4 — Step 3A: Test YOLO bounding-box conversion
# Read-only: no files are created or modified

import pandas as pd

# Select one TRAIN annotation only
sample = annotations.loc[
    annotations["split"] == "train"
].iloc[0]

# Find corresponding M1 image dimensions
image_info = split_manifest.loc[
    split_manifest["image_id"] == sample["image_id"]
].iloc[0]

img_w = float(image_info["width"])
img_h = float(image_info["height"])

assert img_w > 0 and img_h > 0

# Original M1 bounding box
xmin = float(sample["xmin"])
ymin = float(sample["ymin"])
xmax = float(sample["xmax"])
ymax = float(sample["ymax"])

# Convert to normalized YOLO coordinates
x_center = ((xmin + xmax) / 2) / img_w
y_center = ((ymin + ymax) / 2) / img_h
box_width = (xmax - xmin) / img_w
box_height = (ymax - ymin) / img_h

class_id = class_map[sample["class"]]

values = [x_center, y_center, box_width, box_height]

assert all(0 <= v <= 1 for v in values)
assert box_width > 0 and box_height > 0

print("=== M4 YOLO CONVERSION TEST ===")
print("Image ID:", sample["image_id"])
print("Split:", sample["split"])
print("Image dimensions:", img_w, img_h)
print("Original class:", sample["class"])
print("YOLO class ID:", class_id)

print("\nOriginal bounding box:")
print(xmin, ymin, xmax, ymax)

print("\nYOLO annotation:")
print(
    f"{class_id} "
    f"{x_center:.6f} "
    f"{y_center:.6f} "
    f"{box_width:.6f} "
    f"{box_height:.6f}"
)

print("\nPASS: YOLO conversion successful.")


In [ ]:

# M4 — Step 3B: Generate YOLO labels
# Writes only inside the new M4 YOLO directory

from collections import Counter
from pathlib import Path
import numpy as np

required = {"image_id", "split", "class",
            "xmin", "ymin", "xmax", "ymax"}
assert required.issubset(annotations.columns)

# Match annotations to authoritative M1 image dimensions
metadata = split_manifest[
    ["image_id", "split", "width", "height"]
].rename(columns={"split": "manifest_split"})

assert not metadata["image_id"].duplicated().any()

records = annotations.merge(
    metadata,
    on="image_id",
    how="left",
    validate="many_to_one"
)

assert len(records) == len(annotations)
assert records["width"].notna().all()
assert records["height"].notna().all()
assert (records["split"] == records["manifest_split"]).all()
assert set(records["class"]).issubset(class_map)

# Normalize bounding boxes
w = records["width"].astype(float)
h = records["height"].astype(float)

records["xc"] = (
    (records["xmin"] + records["xmax"]) / 2 / w
)
records["yc"] = (
    (records["ymin"] + records["ymax"]) / 2 / h
)
records["bw"] = (
    (records["xmax"] - records["xmin"]) / w
)
records["bh"] = (
    (records["ymax"] - records["ymin"]) / h
)

coords = records[["xc", "yc", "bw", "bh"]].to_numpy()

assert np.isfinite(coords).all()
assert ((coords >= 0) & (coords <= 1)).all()
assert (coords[:, 2:] > 0).all()

# Build all label content in memory before writing
label_files = {}

for (split, image_id), group in records.groupby(
    ["split", "image_id"], sort=False
):
    lines = []

    for _, row in group.iterrows():
      lines.append(
        f"{class_map[row['class']]} "
        f"{row['xc']:.8f} {row['yc']:.8f} "
        f"{row['bw']:.8f} {row['bh']:.8f}"
    )


    # Image IDs must be safe and unique within a split
    assert "/" not in str(image_id)
    assert "\\" not in str(image_id)

    target = yolo_dir / "labels" / split / f"{image_id}.txt"
    assert target not in label_files
    label_files[target] = "\n".join(lines) + "\n"

# Do not overwrite existing label files
existing = [p for p in label_files if p.exists()]
assert not existing, (
    f"{len(existing)} label files already exist. "
    "Stop and inspect before overwriting."
)

for target, content in label_files.items():
    target.write_text(content, encoding="utf-8")

print("=== M4 YOLO LABEL GENERATION ===")
print("Annotations converted:", len(records))
print("Label files created:", len(label_files))

counts = Counter(p.parent.name for p in label_files)
for split in ["train", "val", "test"]:
    print(f"{split}: {counts[split]} label files")

print("\nPASS: YOLO labels generated in M4 workspace.")


In [ ]:

from pathlib import Path

# Use the existing M4 workspace
yolo_dir = project_dir / "M4_Object_Detection" / "data" / "yolo_dataset"

print("=== M4 YOLO LABEL VERIFICATION ===")

for split in ["train", "val", "test"]:
    label_dir = yolo_dir / split / "labels"
    label_files = list(label_dir.glob("*.txt"))

    invalid = 0
    total_boxes = 0

    for file in label_files:
        for line in file.read_text().splitlines():
            parts = line.split()

            if len(parts) != 5:
                invalid += 1
                continue

            try:
                cls = int(parts[0])
                coords = list(map(float, parts[1:]))

                if cls not in range(4) or not all(
                    0 <= x <= 1 for x in coords
                ) or coords[2] <= 0 or coords[3] <= 0:
                    invalid += 1
            except ValueError:
                invalid += 1

            total_boxes += 1

    print(f"\n{split.upper()}")
    print("Label files:", len(label_files))
    print("Bounding boxes:", total_boxes)
    print("Invalid annotations:", invalid)


In [ ]:

from pathlib import Path
import math

yolo_dir = project_dir / "M4_Object_Detection" / "data" / "yolo_dataset"

print("=== M4 YOLO LABEL VERIFICATION ===")

grand_total = 0
grand_invalid = 0

for split in ["train", "val", "test"]:
    label_dir = yolo_dir / "labels" / split
    label_files = list(label_dir.glob("*.txt"))

    total_boxes = 0
    invalid = 0

    for file in label_files:
        for line in file.read_text().splitlines():
            parts = line.split()
            total_boxes += 1

            if len(parts) != 5:
                invalid += 1
                continue

            try:
                cls = int(parts[0])
                coords = list(map(float, parts[1:]))

                if (
                    cls not in range(4)
                    or not all(math.isfinite(x) for x in coords)
                    or not all(0 <= x <= 1 for x in coords)
                    or coords[2] <= 0
                    or coords[3] <= 0
                    or coords[0] - coords[2]/2 < -1e-6
                    or coords[0] + coords[2]/2 > 1 + 1e-6
                    or coords[1] - coords[3]/2 < -1e-6
                    or coords[1] + coords[3]/2 > 1 + 1e-6
                ):
                    invalid += 1
            except ValueError:
                invalid += 1

    grand_total += total_boxes
    grand_invalid += invalid

    print(f"\n{split.upper()}")
    print("Label files:", len(label_files))
    print("Bounding boxes:", total_boxes)
    print("Invalid annotations:", invalid)

print("\n=== FINAL RESULT ===")
print("Total bounding boxes:", grand_total)
print("Invalid annotations:", grand_invalid)

if grand_total == 55002 and grand_invalid == 0:
    print("PASS: YOLO labels verified successfully.")
else:
    print("REVIEW REQUIRED: Counts or coordinates need checking.")


In [ ]:

from pathlib import Path

yolo_dir = project_dir / "M4_Object_Detection" / "data" / "yolo_dataset"

expected = {
    "train": 23016,
    "val": 7207,
    "test": 8161
}

print("=== M4 YOLO IMAGE DIRECTORY CHECK ===")

for split in ["train", "val", "test"]:
    image_dir = yolo_dir / "images" / split
    label_dir = yolo_dir / "labels" / split

    images = [
        p for p in image_dir.iterdir()
        if p.is_file() and p.suffix.lower()
        in {".jpg", ".jpeg", ".png"}
    ]

    labels = list(label_dir.glob("*.txt"))

    print(f"\n{split.upper()}")
    print("Images found:", len(images))
    print("Expected images:", expected[split])
    print("Label files:", len(labels))
    print("Image count matches:", len(images) == expected[split])


## 4. Source archives and image extraction

Extract only the selected images into M4-specific directories. This can take considerable time; do not rerun when the dataset is already present.

In [ ]:

from pathlib import Path
import zipfile

print("=== M4 IMAGE EXTRACTION PRECHECK ===")

# Existing M1 image records
print("Total M1 images:", len(split_manifest))

# Inspect the source archives without extracting
archive_dir = data_dir / "RDD2022_released_through_CRDDC2022" / "RDD2022"

print("\nArchive directory exists:", archive_dir.exists())

if archive_dir.exists():
    for archive in sorted(archive_dir.glob("*.zip")):
        with zipfile.ZipFile(archive) as z:
            images = [
                name for name in z.namelist()
                if name.lower().endswith(
                    (".jpg", ".jpeg", ".png")
                )
            ]

            print(f"{archive.name}: {len(images)} images")

print("\n=== M1 IMAGE ID EXAMPLES ===")

for split in ["train", "val", "test"]:
    subset = split_manifest[
        split_manifest["split"] == split
    ]

    print(f"\n{split.upper()}")
    print(subset[["image_id", "country"]].head(3).to_string(index=False))


In [ ]:

from pathlib import Path
import zipfile

print("=== M4 IMAGE EXTRACTION MAPPING ===")

archive_dir = (
    data_dir
    / "RDD2022_released_through_CRDDC2022"
    / "RDD2022"
)

for archive in sorted(archive_dir.glob("*.zip")):
    with zipfile.ZipFile(archive) as z:
        images = [
            name for name in z.namelist()
            if name.lower().endswith((".jpg", ".jpeg", ".png"))
        ]

        print(f"\nARCHIVE: {archive.name}")
        print("Sample image paths:")

        for name in images[:3]:
            print(" -", name)

print("\n=== M1 SPLIT EXAMPLES ===")

for split in ["train", "val", "test"]:
    subset = split_manifest[
        split_manifest["split"] == split
    ]

    print(f"\n{split.upper()}")
    print(
        subset[
            ["image_id", "country", "source"]
        ].head(3).to_string(index=False)
    )


In [ ]:

from pathlib import Path
from collections import Counter
import zipfile

print("=== M4 IMAGE MAPPING PRECHECK ===")

archive_dir = (
    data_dir
    / "RDD2022_released_through_CRDDC2022"
    / "RDD2022"
)

# Build read-only archive index
archive_index = {}

for archive in sorted(archive_dir.glob("*.zip")):
    with zipfile.ZipFile(archive) as z:
        for member in z.namelist():
            if member.lower().endswith((".jpg", ".jpeg", ".png")):
                name = Path(member).name
                key = f"{archive.stem}_{Path(name).stem}"

                archive_index.setdefault(key, []).append(
                    (archive.name, member)
                )

# Verify each M1 image identifier
missing = []
ambiguous = []
split_counts = Counter()

for row in split_manifest.itertuples(index=False):
    matches = archive_index.get(row.image_id, [])

    if len(matches) == 0:
        missing.append(row.image_id)
    elif len(matches) > 1:
        ambiguous.append(row.image_id)
    else:
        split_counts[row.split] += 1

print("Total M1 images:", len(split_manifest))
print("Unique matches:", sum(split_counts.values()))
print("Missing:", len(missing))
print("Ambiguous:", len(ambiguous))

print("\nMatched images by M1 split:")
for split in ["train", "val", "test"]:
    print(split, split_counts[split])

print("\nSample missing IDs:", missing[:5])
print("Sample ambiguous IDs:", ambiguous[:5])

assert not missing and not ambiguous, (
    "Image mapping requires review before extraction."
)

print("\nPASS: All M1 images uniquely mapped.")


In [ ]:

print("=== M4 NORWAY IMAGE MAPPING FIX ===")

# Add Norway's M1 identifier prefix
with zipfile.ZipFile(archive_dir / "Norway.zip") as z:
    for member in z.namelist():
        if member.lower().endswith((".jpg", ".jpeg", ".png")):
            stem = Path(member).stem
            key = f"Norway-002_{stem}"
            archive_index[key] = [
                ("Norway.zip", member)
            ]

# Recheck all M1 images
missing = []
ambiguous = []
split_counts = Counter()

for row in split_manifest.itertuples(index=False):
    matches = archive_index.get(row.image_id, [])

    if len(matches) == 0:
        missing.append(row.image_id)
    elif len(matches) > 1:
        ambiguous.append(row.image_id)
    else:
        split_counts[row.split] += 1

print("Total M1 images:", len(split_manifest))
print("Unique matches:", sum(split_counts.values()))
print("Missing:", len(missing))
print("Ambiguous:", len(ambiguous))

print("\nMatched images by split:")
for split in ["train", "val", "test"]:
    print(split, split_counts[split])

assert len(missing) == 0
assert len(ambiguous) == 0

print("\nPASS: All M1 images uniquely mapped.")


In [ ]:

from pathlib import Path
from collections import Counter
import zipfile
import shutil
import time

print("=== M4 YOLO IMAGE EXTRACTION ===")

yolo_dir = project_dir / "M4_Object_Detection" / "data" / "yolo_dataset"

expected = split_manifest["split"].value_counts().to_dict()
copied = Counter()
start = time.time()

# Group images by source archive
jobs = {}

for row in split_manifest.itertuples(index=False):
    archive_name, member = archive_index[row.image_id][0]

    destination = (
        yolo_dir / "images" / row.split /
        f"{row.image_id}{Path(member).suffix.lower()}"
    )

    jobs.setdefault(archive_name, []).append(
        (member, destination)
    )

# Extract only M1-selected images
for archive_name, items in jobs.items():
    print(f"\nProcessing {archive_name}...")

    with zipfile.ZipFile(archive_dir / archive_name) as z:
        for member, destination in items:
            destination.parent.mkdir(
                parents=True, exist_ok=True
            )

            # Avoid rewriting completed files
            if not destination.exists():
                with z.open(member) as src:
                    with open(destination, "wb") as dst:
                        shutil.copyfileobj(src, dst)

            copied[destination.parent.name] += 1

    print(f"Completed: {archive_name}")

print("\n=== EXTRACTION SUMMARY ===")

for split in ["train", "val", "test"]:
    print(
        f"{split}: {copied[split]} / "
        f"{expected.get(split, 0)}"
    )

print(f"\nElapsed: {(time.time()-start)/60:.1f} minutes")

assert all(
    copied[s] == expected[s]
    for s in ["train", "val", "test"]
)

print("\nPASS: M4 image extraction completed.")


In [ ]:

from pathlib import Path

print("=== M4 FINAL IMAGE AND LABEL CHECK ===")

image_extensions = {".jpg", ".jpeg", ".png"}

for split in ["train", "val", "test"]:
    image_dir = yolo_dir / "images" / split
    label_dir = yolo_dir / "labels" / split

    images = [
        p for p in image_dir.iterdir()
        if p.is_file() and p.suffix.lower() in image_extensions
    ]

    labels = list(label_dir.glob("*.txt"))

    image_ids = {p.stem for p in images}
    label_ids = {p.stem for p in labels}

    expected = int(
        (split_manifest["split"] == split).sum()
    )

    missing_images = expected - len(images)
    unmatched_labels = label_ids - image_ids
    unlabeled_images = image_ids - label_ids

    print(f"\n{split.upper()}")
    print("Images:", len(images))
    print("Expected images:", expected)
    print("YOLO label files:", len(labels))
    print("Missing image count:", missing_images)
    print("Labels without images:", len(unmatched_labels))
    print("Images without label files:", len(unlabeled_images))

    assert len(images) == expected
    assert len(unmatched_labels) == 0

print("\nPASS: Image counts and label references verified.")


## 5. Annotation coverage and background policy

Investigate images without supported damage annotations. A missing supported label does not by itself establish that an image is truly damage-free.

In [ ]:

print("=== M4 UNLABELED IMAGE INVESTIGATION ===")

# M1 image identifiers with at least one annotation
annotated_ids = set(annotations["image_id"])

for split in ["train", "val", "test"]:
    subset = split_manifest[
        split_manifest["split"] == split
    ]

    image_ids = set(subset["image_id"])

    expected_unlabeled = image_ids - annotated_ids

    label_dir = yolo_dir / "labels" / split
    actual_labeled = {
        p.stem for p in label_dir.glob("*.txt")
    }

    actual_unlabeled = image_ids - actual_labeled

    print(f"\n{split.upper()}")
    print("Images without M1 annotations:",
          len(expected_unlabeled))
    print("Images without YOLO labels:",
          len(actual_unlabeled))
    print("Unexpected missing YOLO labels:",
          len(actual_unlabeled - expected_unlabeled))
    print("Unexpected YOLO labels:",
          len(expected_unlabeled - actual_unlabeled))

    assert actual_unlabeled == expected_unlabeled, (
        f"Mismatch in {split}"
    )

print("\nPASS: YOLO label coverage matches M1 annotations.")


In [ ]:

print("=== M4 ANNOTATION COVERAGE BY COUNTRY ===")

annotated_ids = set(annotations["image_id"])

coverage = split_manifest[
    ["image_id", "country", "split"]
].copy()

coverage["has_annotation"] = (
    coverage["image_id"].isin(annotated_ids)
)

summary = (
    coverage.groupby(["country", "split"])
    .agg(
        total_images=("image_id", "size"),
        annotated_images=("has_annotation", "sum")
    )
    .reset_index()
)

summary["unannotated_images"] = (
    summary["total_images"] -
    summary["annotated_images"]
)

summary["annotation_coverage_pct"] = (
    100 * summary["annotated_images"] /
    summary["total_images"]
).round(2)

print(summary.to_string(index=False))

print("\nPASS: Coverage analysis completed.")


In [ ]:
# Inspect training images without bounding-box annotations
train = split_manifest[split_manifest["split"] == "train"].copy()

annotated_ids = set(annotations["image_id"])
unannotated_train = train[
    ~train["image_id"].isin(annotated_ids)
]

print("Unannotated training images:", len(unannotated_train))
print("\nXML path availability:")
print(unannotated_train["xml_path"].notna().value_counts())

print("\nSample XML paths:")
print(unannotated_train[
    ["image_id", "country", "xml_path"]
].head(5).to_string(index=False))

In [ ]:
import zipfile
from pathlib import Path

# Original RDD2022 archives
zip_dir = (
    project_dir
    / "Data"
    / "RDD2022_released_through_CRDDC2022"
    / "RDD2022"
)

# Inspect only India (training split)
india_zip = zip_dir / "India.zip"

print("India ZIP exists:", india_zip.exists())

if india_zip.exists():
    with zipfile.ZipFile(india_zip) as z:
        xml_files = [
            name for name in z.namelist()
            if name.lower().endswith(".xml")
        ]

        print("Total XML files:", len(xml_files))
        print("\nFirst 5 XML paths:")
        for name in xml_files[:5]:
            print(name)

In [ ]:
import zipfile
import xml.etree.ElementTree as ET

# Inspect five unannotated India training images
samples = unannotated_train[
    unannotated_train["country"] == "India"
].head(5)

with zipfile.ZipFile(india_zip) as z:
    for _, row in samples.iterrows():
        xml_name = row["xml_path"].replace("\\", "/").split("/")[-1]
        matches = [
            name for name in z.namelist()
            if name.endswith("/" + xml_name)
        ]

        if len(matches) != 1:
            print(row["image_id"], "XML matches:", len(matches))
            continue

        xml_root = ET.fromstring(z.read(matches[0]))
        objects = xml_root.findall(".//object")

        print(row["image_id"], "| XML objects:", len(objects))

In [ ]:
import zipfile
import xml.etree.ElementTree as ET
from collections import Counter

zip_names = {
    "India": "India.zip",
    "Japan": "Japan.zip",
    "United_States": "United_States.zip"
}

results = Counter()
missing = []

for country, zip_name in zip_names.items():
    rows = unannotated_train[
        unannotated_train["country"] == country
    ]

    with zipfile.ZipFile(zip_dir / zip_name) as z:
        xml_lookup = {
            name.split("/")[-1]: name
            for name in z.namelist()
            if name.lower().endswith(".xml")
        }

        for _, row in rows.iterrows():
            xml_name = row["xml_path"].replace("\\", "/").split("/")[-1]
            archive_path = xml_lookup.get(xml_name)

            if archive_path is None:
                missing.append(row["image_id"])
                results[(country, "missing_xml")] += 1
                continue

            root_xml = ET.fromstring(z.read(archive_path))
            object_count = len(root_xml.findall(".//object"))

            if object_count == 0:
                results[(country, "zero_objects")] += 1
            else:
                results[(country, "has_objects")] += 1

print("=== UNANNOTATED TRAINING XML AUDIT ===")
for country in zip_names:
    print(
        country,
        "| Zero objects:", results[(country, "zero_objects")],
        "| Has objects:", results[(country, "has_objects")],
        "| Missing XML:", results[(country, "missing_xml")]
    )

print("\nTotal checked:", sum(results.values()))
print("Expected:", len(unannotated_train))

In [ ]:
from collections import Counter
import zipfile
import xml.etree.ElementTree as ET

class_counts = Counter()
examples = []

for country, zip_name in {
    "India": "India.zip",
    "Japan": "Japan.zip"
}.items():

    rows = unannotated_train[
        unannotated_train["country"] == country
    ]

    with zipfile.ZipFile(zip_dir / zip_name) as z:
        lookup = {
            name.split("/")[-1]: name
            for name in z.namelist()
            if name.lower().endswith(".xml")
        }

        for _, row in rows.iterrows():
            xml_name = row["xml_path"].replace("\\", "/").split("/")[-1]
            root_xml = ET.fromstring(z.read(lookup[xml_name]))

            objects = root_xml.findall(".//object")

            for obj in objects:
                name = obj.findtext("name", default="UNKNOWN")
                class_counts[(country, name)] += 1

            if objects and len(examples) < 10:
                examples.append((
                    row["image_id"],
                    country,
                    [obj.findtext("name") for obj in objects]
                ))

print("=== ORIGINAL XML OBJECT CLASSES ===")

for (country, name), count in sorted(class_counts.items()):
    print(country, "|", name, "|", count)

print("\n=== EXAMPLE IMAGES ===")
for example in examples:
    print(example)

In [ ]:
supported = {"D00", "D10", "D20", "D40"}

supported_found = []
unsupported_only = 0
zero_objects = 0

for country, zip_name in {
    "India": "India.zip",
    "Japan": "Japan.zip"
}.items():

    rows = unannotated_train[
        unannotated_train["country"] == country
    ]

    with zipfile.ZipFile(zip_dir / zip_name) as z:
        lookup = {
            name.split("/")[-1]: name
            for name in z.namelist()
            if name.lower().endswith(".xml")
        }

        for _, row in rows.iterrows():
            xml_name = row["xml_path"].replace("\\", "/").split("/")[-1]
            root_xml = ET.fromstring(z.read(lookup[xml_name]))

            classes = {
                (obj.findtext("name") or "").strip()
                for obj in root_xml.findall(".//object")
            }

            if not classes:
                zero_objects += 1
            elif classes & supported:
                supported_found.append((row["image_id"], classes))
            else:
                unsupported_only += 1

print("Zero-object XML images:", zero_objects)
print("Unsupported-class-only images:", unsupported_only)
print("Images containing supported classes:", len(supported_found))
print("Examples:", supported_found[:5])

In [ ]:
val = split_manifest[split_manifest["split"] == "val"].copy()

unannotated_val = val[
    ~val["image_id"].isin(set(annotations["image_id"]))
]

print("=== UNANNOTATED VALIDATION IMAGES ===")
print("Total:", len(unannotated_val))
print("\nBy country:")
print(unannotated_val["country"].value_counts())

print("\nXML paths recorded:")
print(unannotated_val["xml_path"].notna().value_counts())

In [ ]:
import zipfile
import xml.etree.ElementTree as ET
from collections import Counter

supported = {"D00", "D10", "D20", "D40"}

val_zip_names = {
    "Czech": "Czech.zip",
    "China_Drone": "China_Drone.zip",
    "China_MotorBike": "China_MotorBike.zip"
}

val_results = Counter()
val_class_counts = Counter()
val_supported_examples = []

for country, zip_name in val_zip_names.items():

    rows = unannotated_val[
        unannotated_val["country"] == country
    ]

    with zipfile.ZipFile(zip_dir / zip_name) as z:

        xml_lookup = {
            name.split("/")[-1]: name
            for name in z.namelist()
            if name.lower().endswith(".xml")
        }

        for _, row in rows.iterrows():

            xml_name = row["xml_path"].replace("\\", "/").split("/")[-1]
            archive_path = xml_lookup.get(xml_name)

            if archive_path is None:
                val_results[(country, "missing_xml")] += 1
                continue

            root_xml = ET.fromstring(z.read(archive_path))

            classes = [
                (obj.findtext("name") or "").strip()
                for obj in root_xml.findall(".//object")
            ]

            if not classes:
                val_results[(country, "zero_objects")] += 1

            elif set(classes) & supported:
                val_results[(country, "supported_found")] += 1

                if len(val_supported_examples) < 5:
                    val_supported_examples.append(
                        (row["image_id"], classes)
                    )

            else:
                val_results[(country, "unsupported_only")] += 1

            for cls in classes:
                val_class_counts[(country, cls)] += 1

print("=== VALIDATION XML AUDIT ===")

for country in val_zip_names:
    print(
        country,
        "| Zero objects:", val_results[(country, "zero_objects")],
        "| Unsupported only:", val_results[(country, "unsupported_only")],
        "| Supported found:", val_results[(country, "supported_found")],
        "| Missing XML:", val_results[(country, "missing_xml")]
    )

print("\n=== ORIGINAL XML CLASSES ===")
for key, count in sorted(val_class_counts.items()):
    print(key, count)

print("\nSupported-class examples:", val_supported_examples)

In [ ]:
# M4 proposed inclusion plan — counts only, no file changes

m4_plan = {
    "train": {
        "annotated": 15927,
        "zero_object": 4715,
        "unsupported_only": 2374
    },
    "val": {
        "annotated": 4925,
        "zero_object": 1762,
        "unsupported_only": 520
    }
}

for split, counts in m4_plan.items():
    included = counts["annotated"] + counts["zero_object"]
    excluded = counts["unsupported_only"]
    original = included + excluded

    print(f"\n=== {split.upper()} ===")
    print("Original images:", original)
    print("Proposed included:", included)
    print("Proposed excluded:", excluded)
    print("Coverage:", f"{included/original*100:.2f}%")

In [ ]:
# Check whether annotated images also contain unsupported XML classes
import zipfile
import xml.etree.ElementTree as ET
from collections import Counter

supported = {"D00", "D10", "D20", "D40"}

archives = {
    "India": "India.zip",
    "Japan": "Japan.zip",
    "United_States": "United_States.zip",
    "Czech": "Czech.zip",
    "China_Drone": "China_Drone.zip",
    "China_MotorBike": "China_MotorBike.zip"
}

counts = Counter()
examples = []

for country, zip_name in archives.items():
    rows = split_manifest[
        (split_manifest["country"] == country) &
        (split_manifest["image_id"].isin(annotated_ids))
    ]

    with zipfile.ZipFile(zip_dir / zip_name) as z:
        lookup = {
            name.split("/")[-1]: name
            for name in z.namelist()
            if name.lower().endswith(".xml")
        }

        for _, row in rows.iterrows():
            xml_name = row["xml_path"].replace("\\", "/").split("/")[-1]
            path = lookup.get(xml_name)

            if path is None:
                counts[(country, "missing_xml")] += 1
                continue

            root_xml = ET.fromstring(z.read(path))
            classes = {
                (obj.findtext("name") or "").strip()
                for obj in root_xml.findall(".//object")
            }

            if classes & supported and classes - supported:
                counts[(country, "mixed")] += 1
                if len(examples) < 5:
                    examples.append((row["image_id"], sorted(classes)))
            elif classes & supported:
                counts[(country, "supported_only")] += 1
            else:
                counts[(country, "no_supported")] += 1

print("=== MIXED-CLASS XML AUDIT ===")
for country in archives:
    print(country, dict(
        (key[1], value)
        for key, value in counts.items()
        if key[0] == country
    ))

print("\nMixed-class examples:", examples)

In [ ]:
# M4 inclusion policy summary — no files modified

train_annotated = 15927
train_background = 4715
train_unsupported = 2374

val_annotated = 4925
val_background = 1762
val_unsupported = 520

print("=== PROPOSED M4 DATASET ===")
print("Training:", train_annotated + train_background)
print("Validation:", val_annotated + val_background)

print("\nExcluded unsupported-only images:")
print("Training:", train_unsupported)
print("Validation:", val_unsupported)

print("\nMixed-class images retained:")
print("Training:", 494 + 3767)
print("Validation:", 105 + 180)

assert train_annotated + train_background + train_unsupported == 23016
assert val_annotated + val_background + val_unsupported == 7207

print("\nPASS: Original split totals preserved.")

In [ ]:
# Build M4-specific inclusion lists without changing any files

from pathlib import Path
import zipfile
import xml.etree.ElementTree as ET

supported = {"D00", "D10", "D20", "D40"}

archive_names = {
    "India": "India.zip",
    "Japan": "Japan.zip",
    "United_States": "United_States.zip",
    "Czech": "Czech.zip",
    "China_Drone": "China_Drone.zip",
    "China_MotorBike": "China_MotorBike.zip"
}

m4_included = {"train": set(), "val": set()}
m4_excluded = {"train": set(), "val": set()}

for country, archive_name in archive_names.items():
    rows = split_manifest[
        (split_manifest["country"] == country) &
        (split_manifest["split"].isin(["train", "val"]))
    ]

    with zipfile.ZipFile(zip_dir / archive_name) as z:
        xml_lookup = {
            Path(name).name: name
            for name in z.namelist()
            if name.lower().endswith(".xml")
        }

        for _, row in rows.iterrows():
            xml_name = str(row["xml_path"]).replace("\\", "/").split("/")[-1]
            archive_path = xml_lookup.get(xml_name)

            if archive_path is None:
                raise FileNotFoundError(f"Missing XML: {row['image_id']}")

            root_xml = ET.fromstring(z.read(archive_path))
            classes = {
                (obj.findtext("name") or "").strip()
                for obj in root_xml.findall(".//object")
            }

            split = row["split"]
            image_id = row["image_id"]

            # Retain supported-class images and zero-object backgrounds
            if not classes or classes & supported:
                m4_included[split].add(image_id)
            else:
                m4_excluded[split].add(image_id)

print("=== M4 INCLUSION LISTS ===")
for split in ["train", "val"]:
    print(
        split,
        "| Included:", len(m4_included[split]),
        "| Excluded:", len(m4_excluded[split])
    )

assert len(m4_included["train"]) == 20642
assert len(m4_included["val"]) == 6687
assert len(m4_excluded["train"]) == 2374
assert len(m4_excluded["val"]) == 520

assert not (m4_included["train"] & m4_included["val"])

print("\nPASS: Inclusion lists verified. No files modified.")

In [ ]:
import pandas as pd

# Create a reproducible M4-only inclusion manifest
m4_manifest = split_manifest[
    split_manifest["split"].isin(["train", "val"])
][["image_id", "country", "split"]].copy()

m4_manifest["include_m4"] = m4_manifest.apply(
    lambda row: row["image_id"] in m4_included[row["split"]],
    axis=1
)

m4_manifest["reason"] = m4_manifest["include_m4"].map({
    True: "Included: supported annotation or zero-object XML",
    False: "Excluded: unsupported classes only"
})

# Save separately from the original M1 files
output_path = m4_dir / "m4_inclusion_manifest.csv"
m4_manifest.to_csv(output_path, index=False)

print("Saved:", output_path)
print("\nInclusion summary:")
print(
    pd.crosstab(
        m4_manifest["split"],
        m4_manifest["include_m4"]
    )
)

assert len(m4_manifest) == 30223
assert m4_manifest["include_m4"].sum() == 27329

print("\nPASS: M4 inclusion manifest saved.")

## 6. Dataset checks and YAML configuration

Check image-label correspondence and class mapping: D00, D10, D20, D40.

In [ ]:
from pathlib import Path

train_dir = yolo_dir / "images" / "train"

expected_ids = set(
    split_manifest.loc[
        split_manifest["split"] == "train", "image_id"
    ]
)

actual_ids = {
    p.stem
    for p in train_dir.glob("*.jpg")
}

missing = expected_ids - actual_ids
unexpected = actual_ids - expected_ids

print("=== M4 TRAINING IMAGE VERIFICATION ===")
print("Expected images:", len(expected_ids))
print("Actual images:", len(actual_ids))
print("Missing images:", len(missing))
print("Unexpected images:", len(unexpected))

assert len(expected_ids) == 23016
assert not missing, f"Missing images: {list(missing)[:5]}"
assert not unexpected, f"Unexpected images: {list(unexpected)[:5]}"

print("\nPASS: All training images verified.")

In [ ]:
import pandas as pd

# Load saved M4 inclusion decisions
m4_manifest = pd.read_csv(
    m4_dir / "m4_inclusion_manifest.csv"
)

list_dir = m4_dir / "data" / "experiment_lists"
list_dir.mkdir(parents=True, exist_ok=True)

for split, expected in [("train", 20642), ("val", 6687)]:

    selected = m4_manifest[
        (m4_manifest["split"] == split) &
        (m4_manifest["include_m4"] == True)
    ]

    paths = [
        str(yolo_dir / "images" / split / f"{image_id}.jpg")
        for image_id in selected["image_id"]
    ]

    missing = [p for p in paths if not Path(p).is_file()]

    assert len(paths) == expected
    assert not missing, f"Missing images: {missing[:5]}"

    output = list_dir / f"{split}.txt"
    output.write_text("\n".join(paths) + "\n")

    print(f"{split}: {len(paths)} images")
    print("Saved:", output)

print("\nPASS: M4 experiment image lists created.")

In [ ]:
from pathlib import Path

print("=== YOLO IMAGE-TO-LABEL PATH CHECK ===")

for split in ["train", "val"]:
    list_path = list_dir / f"{split}.txt"

    paths = [
        line.strip()
        for line in list_path.read_text().splitlines()
        if line.strip()
    ]

    print(f"\n{split.upper()}")

    for image_str in paths[:3]:
        image_path = Path(image_str)

        label_str = image_str.replace(
            "/images/", "/labels/"
        ).rsplit(".", 1)[0] + ".txt"

        label_path = Path(label_str)

        print("Image exists:", image_path.is_file())
        print("Label exists:", label_path.is_file())
        print("Image:", image_path.name)
        print("Label:", label_path.name)

    print("Total selected images:", len(paths))

In [ ]:
# M4 — Full YOLO image-label consistency check

from pathlib import Path

annotated_ids = set(annotations["image_id"])

print("=== M4 FULL IMAGE-LABEL VERIFICATION ===")

for split, expected in [("train", 20642), ("val", 6687)]:

    selected = m4_manifest[
        (m4_manifest["split"] == split) &
        (m4_manifest["include_m4"])
    ]

    missing_images = []
    missing_labels = []
    background_count = 0
    annotated_count = 0

    for image_id in selected["image_id"]:

        image_path = yolo_dir / "images" / split / f"{image_id}.jpg"
        label_path = yolo_dir / "labels" / split / f"{image_id}.txt"

        if not image_path.is_file():
            missing_images.append(image_id)

        if image_id in annotated_ids:
            annotated_count += 1

            if not label_path.is_file():
                missing_labels.append(image_id)
        else:
            background_count += 1

    print(f"\n{split.upper()}")
    print("Selected images:", len(selected))
    print("Annotated images:", annotated_count)
    print("Background images:", background_count)
    print("Missing images:", len(missing_images))
    print("Missing required labels:", len(missing_labels))

    assert len(selected) == expected
    assert not missing_images, missing_images[:5]
    assert not missing_labels, missing_labels[:5]

print("\nPASS: All selected images and required labels verified.")

In [ ]:
from pathlib import Path

# M4 — Four-class YOLO dataset configuration

config_path = m4_dir / "data" / "m4_yolo.yaml"

yaml_content = f"""\
path: "{yolo_dir}"
train: "{list_dir / 'train.txt'}"
val: "{list_dir / 'val.txt'}"

nc: 4
names:
  0: D00
  1: D10
  2: D20
  3: D40
"""

config_path.write_text(yaml_content, encoding="utf-8")

print("=== M4 YOLO CONFIGURATION ===")
print("Saved:", config_path)
print()
print(config_path.read_text())

assert (list_dir / "train.txt").is_file()
assert (list_dir / "val.txt").is_file()

print("PASS: YOLO configuration created.")

## 7. Colab local training dataset

Local `/content/m4_local` data is temporary and must be reconstructed after a runtime reset. The persistent dataset is on Google Drive.

In [ ]:

from pathlib import Path

p = Path("/content/m4_local/yolo_dataset")

print("=== EXACT LOCAL DATASET CHECK ===")
print("Dataset exists:", p.exists())

for split in ["train", "val"]:
    for folder in ["images", "labels"]:
        d = p / folder / split
        print(f"{folder}/{split}:",
              sum(1 for _ in d.iterdir()) if d.exists() else 0)


In [ ]:
from pathlib import Path
import shutil
import time

source = Path(
    "/content/drive/MyDrive/AI Project /M4_Object_Detection/data/yolo_dataset"
)
destination = Path("/content/m4_local/yolo_dataset")

expected = {
    ("images", "train"): 23016,
    ("labels", "train"): 15927,
    ("images", "val"): 7207,
    ("labels", "val"): 4925,
}

start = time.time()

for (folder, split), total in expected.items():
    src = source / folder / split
    dst = destination / folder / split

    assert src.is_dir(), f"Source missing: {src}"
    dst.mkdir(parents=True, exist_ok=True)

    print(f"\nChecking {folder}/{split}...", flush=True)

    copied = 0
    skipped = 0

    for file in src.iterdir():
        if not file.is_file():
            continue

        target = dst / file.name

        if target.exists() and target.stat().st_size == file.stat().st_size:
            skipped += 1
            continue

        shutil.copy2(file, target)
        copied += 1

        if copied % 1000 == 0:
            print(f"Copied {copied:,} new files...", flush=True)

    actual = sum(1 for p in dst.iterdir() if p.is_file())

    print(f"Completed: {actual:,}/{total:,}")
    print(f"Skipped existing: {skipped:,} | Copied: {copied:,}")

    assert actual == total, f"Count mismatch: {folder}/{split}"

print(f"\nPASS: Local M4 dataset complete in {(time.time()-start)/60:.1f} minutes.")

In [ ]:

from pathlib import Path

root = Path("/content/m4_local/yolo_dataset")

expected = {
    "images/train": 23016,
    "labels/train": 15927,
    "images/val": 7207,
    "labels/val": 4925,
}

for folder, total in expected.items():
    path = root / folder
    actual = sum(1 for f in path.iterdir() if f.is_file()) if path.exists() else 0
    print(f"{folder}: {actual:,} / {total:,}")
    assert actual == total, f"Mismatch in {folder}"

print("\nPASS: All M4 training and validation files are present.")


In [ ]:

from pathlib import Path

# Permanent M4 training outputs in Google Drive
m4_results = Path(
    "/content/drive/MyDrive/AI Project /M4_Object_Detection/runs"
)

m4_results.mkdir(parents=True, exist_ok=True)

# Confirm the destination is available
assert m4_results.is_dir(), "Results folder unavailable"

print("PASS: M4 results folder is ready")
print("Training outputs will be saved to:", m4_results)

# Confirm local training data remains available
local_data = Path("/content/m4_local/yolo_dataset")
assert (local_data / "images/train").is_dir()
assert (local_data / "images/val").is_dir()

print("PASS: Local training and validation dataset available")


In [ ]:

from pathlib import Path
import yaml

local_data = Path("/content/m4_local/yolo_dataset")
config_path = local_data / "data.yaml"

config = {
    "path": str(local_data),
    "train": "images/train",
    "val": "images/val",
    "nc": 4,
    "names": {
        0: "D00",
        1: "D10",
        2: "D20",
        3: "D40",
    },
}

with open(config_path, "w") as f:
    yaml.safe_dump(config, f, sort_keys=False)

# Preserve a permanent copy of the configuration
drive_config = m4_results.parent / "data.yaml"
with open(drive_config, "w") as f:
    yaml.safe_dump(config, f, sort_keys=False)

print("PASS: YOLO configuration created")
print("Local config:", config_path)
print("Drive backup:", drive_config)
print(yaml.safe_dump(config, sort_keys=False))


## 8. Training environment and model configuration

YOLO11n baseline: 30 epochs, 640-pixel input, batch size 16, seed 42, deterministic training. Norway must not be used for model selection.

In [ ]:

import torch
import importlib.util

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

installed = importlib.util.find_spec("ultralytics") is not None
print("Ultralytics installed:", installed)

assert torch.cuda.is_available(), "GPU is not available"
print("\nPASS: GPU environment ready")


In [ ]:

%pip install -q ultralytics

from ultralytics import YOLO
import ultralytics

print("Ultralytics version:", ultralytics.__version__)
print("PASS: YOLO installed successfully")


In [ ]:

from ultralytics import YOLO
from pathlib import Path
import torch

# Confirm the prepared dataset is still available
data_yaml = Path("/content/m4_local/yolo_dataset/data.yaml")
assert data_yaml.is_file(), "YOLO configuration missing"

assert torch.cuda.is_available(), "GPU unavailable"

# Load a small pretrained YOLO detection model
model = YOLO("yolo11n.pt")

print("Model:", model.model.yaml["yaml_file"] if "yaml_file" in model.model.yaml else "YOLO11n")
print("Task:", model.task)
print("GPU:", torch.cuda.get_device_name(0))
print("Dataset:", data_yaml)
print("\nPASS: YOLO model loaded successfully")


In [ ]:

from pathlib import Path

training_config = {
    "data": "/content/m4_local/yolo_dataset/data.yaml",
    "epochs": 30,
    "imgsz": 640,
    "batch": 16,
    "device": 0,
    "workers": 2,
    "seed": 42,
    "deterministic": True,
    "patience": 10,
    "save": True,
    "save_period": 5,
    "project": str(m4_results),
    "name": "yolo11n_baseline",
    "exist_ok": False,
    "plots": True,
}

assert Path(training_config["data"]).is_file()
assert Path(training_config["project"]).is_dir()

print("PASS: Training configuration ready")
for key, value in training_config.items():
    print(f"{key}: {value}")


## 9. Smoke test and baseline training

The one-epoch smoke test is separate from the 30-epoch baseline. **Do not run these cells just to review the notebook**: they start training.

In [ ]:

from ultralytics import YOLO

# Separate test run; does not overwrite the baseline
smoke_model = YOLO("yolo11n.pt")

smoke_config = training_config.copy()
smoke_config.update({
    "epochs": 1,
    "name": "yolo11n_smoke_test",
    "patience": 0,
    "save_period": -1,
    "plots": False,
})

smoke_results = smoke_model.train(**smoke_config)

print("\nPASS: YOLO smoke test completed")
print("Results saved to:", smoke_model.trainer.save_dir)


In [ ]:

from pathlib import Path

run_dir = m4_results / "yolo11n_smoke_test"

for name in ["best.pt", "last.pt"]:
    checkpoint = run_dir / "weights" / name
    print(f"{name}: {'SAVED' if checkpoint.is_file() else 'MISSING'}")
    assert checkpoint.is_file(), f"Missing: {name}"

print("\nPASS: YOLO checkpoints safely stored in Google Drive")


In [ ]:

from ultralytics import YOLO

# Start a fresh baseline from pretrained weights
baseline_model = YOLO("yolo11n.pt")

# Use the previously verified training settings
baseline_results = baseline_model.train(**training_config)

print("\nPASS: M4 baseline training completed")
print("Results saved to:", baseline_model.trainer.save_dir)


In [ ]:

from pathlib import Path

run_dir = Path(
    "/content/drive/MyDrive/AI Project /M4_Object_Detection/runs/yolo11n_baseline"
)

for filename in ["best.pt", "last.pt"]:
    path = run_dir / "weights" / filename
    assert path.is_file(), f"Missing: {path}"
    print(f"{filename}: SAVED ({path.stat().st_size / 1e6:.2f} MB)")

print("\nPASS: M4 baseline checkpoints preserved in Google Drive")


## 10. Training curves and visualizations

Display existing training artifacts from Drive; figures are not embedded in the uploaded notebook.

In [ ]:

from pathlib import Path
from IPython.display import display, Image

run_dir = Path(
    "/content/drive/MyDrive/AI Project /M4_Object_Detection/runs/yolo11n_baseline"
)

results_plot = run_dir / "results.png"

assert results_plot.is_file(), f"Missing: {results_plot}"

display(Image(filename=str(results_plot), width=1100))

print("PASS: Training curves displayed successfully")


In [ ]:

from pathlib import Path
from IPython.display import display, Image

run_dir = Path(
    "/content/drive/MyDrive/AI Project /M4_Object_Detection/runs/yolo11n_baseline"
)

matrix_path = run_dir / "confusion_matrix_normalized.png"

assert matrix_path.is_file(), f"Missing: {matrix_path}"

display(Image(filename=str(matrix_path), width=950))

print("PASS: Normalized confusion matrix displayed")


In [ ]:

from pathlib import Path
from IPython.display import display, Image

run_dir = Path(
    "/content/drive/MyDrive/AI Project /M4_Object_Detection/runs/yolo11n_baseline"
)

for filename in [
    "val_batch0_labels.jpg",
    "val_batch0_pred.jpg"
]:
    path = run_dir / filename
    assert path.is_file(), f"Missing: {path}"
    print(f"\n{filename}")
    display(Image(filename=str(path), width=1100))

print("\nPASS: Validation labels and predictions displayed")


In [ ]:

from pathlib import Path
from IPython.display import display, Image

run_dir = Path(
    "/content/drive/MyDrive/AI Project /M4_Object_Detection/runs/yolo11n_baseline"
)

for filename in [
    "val_batch1_labels.jpg",
    "val_batch1_pred.jpg"
]:
    path = run_dir / filename
    assert path.is_file(), f"Missing: {path}"
    print(f"\n{filename}")
    display(Image(filename=str(path), width=1100))

print("\nPASS: Second validation batch displayed")


In [ ]:

from pathlib import Path
from IPython.display import display, Image

run_dir = Path(
    "/content/drive/MyDrive/AI Project /M4_Object_Detection/runs/yolo11n_baseline"
)

labels_path = run_dir / "val_batch1_labels.jpg"

assert labels_path.is_file(), f"Missing: {labels_path}"

display(Image(filename=str(labels_path), width=1200))

print("PASS: China_Drone ground-truth labels displayed")


In [ ]:

from pathlib import Path
from IPython.display import display, Image

run_dir = Path(
    "/content/drive/MyDrive/AI Project /M4_Object_Detection/runs/yolo11n_baseline"
)

for filename in [
    "val_batch2_labels.jpg",
    "val_batch2_pred.jpg"
]:
    path = run_dir / filename
    assert path.is_file(), f"Missing: {path}"
    print(f"\n{filename}")
    display(Image(filename=str(path), width=1100))

print("\nPASS: Third validation batch displayed")


In [ ]:

from pathlib import Path
from IPython.display import display, Image

run_dir = Path(
    "/content/drive/MyDrive/AI Project /M4_Object_Detection/runs/yolo11n_baseline"
)

labels_path = run_dir / "val_batch2_labels.jpg"

assert labels_path.is_file(), f"Missing: {labels_path}"

display(Image(filename=str(labels_path), width=1100))

print("PASS: Third validation batch ground-truth labels displayed")


In [ ]:

from pathlib import Path
import torch

base = Path("/content/drive/MyDrive/AI Project /M4_Object_Detection")

checks = {
    "Best model checkpoint": base / "runs/yolo11n_baseline/weights/best.pt",
    "Validation images": Path("/content/m4_local/yolo_dataset/images/val"),
    "Validation labels": Path("/content/m4_local/yolo_dataset/labels/val"),
    "Dataset configuration": Path("/content/m4_local/yolo_dataset/data.yaml"),
}

print("GPU available:", torch.cuda.is_available())

for name, path in checks.items():
    status = "PASS" if path.exists() else "NOT FOUND"
    print(f"{name}: {status}")


In [ ]:

from pathlib import Path
import ultralytics

base = Path("/content/drive/MyDrive/AI Project /M4_Object_Detection")
val_dir = base / "data/yolo_dataset/images/val"

print("Ultralytics version:", ultralytics.__version__)

images = [
    p for p in val_dir.iterdir()
    if p.suffix.lower() in {".jpg", ".jpeg", ".png"}
]

print("Total validation images:", len(images))

for source in ["China_Drone", "China_MotorBike", "Czech"]:
    count = sum(p.name.startswith(source) for p in images)
    print(f"{source}: {count}")

print("\nPASS: Validation dataset inspected")


In [ ]:

from pathlib import Path
from collections import Counter
import ultralytics

base = Path("/content/drive/MyDrive/AI Project /M4_Object_Detection")
val_dir = base / "data/yolo_dataset/images/val"

assert val_dir.is_dir(), "Validation directory not found"

images = [
    p for p in val_dir.iterdir()
    if p.suffix.lower() in {".jpg", ".jpeg", ".png"}
]

sources = ["China_Drone", "China_MotorBike", "Czech"]

print("Ultralytics version:", ultralytics.__version__)
print("Total validation images:", len(images))

for source in sources:
    count = sum(p.name.startswith(source + "_") for p in images)
    print(f"{source}: {count:,}")

print("\nPASS: Validation dataset inspected")


## 11. Validation by source

Validation is based on the M1 validation sources, not the Norway test set. Preserve the existing source-specific evaluation.

In [ ]:

from ultralytics import YOLO
from pathlib import Path

base = Path("/content/drive/MyDrive/AI Project /M4_Object_Detection")

best_model_path = (
    base / "runs/yolo11n_baseline/weights/best.pt"
)

assert best_model_path.is_file(), "Best checkpoint not found"

model = YOLO(str(best_model_path))

print("PASS: Best YOLO11n checkpoint loaded")
print("Model classes:", model.names)
print("Checkpoint:", best_model_path)


In [ ]:

from pathlib import Path
import yaml

base = Path("/content/drive/MyDrive/AI Project /M4_Object_Detection")
dataset = base / "data/yolo_dataset"
source = "China_Drone"

# Verify that the source-specific images exist
val_images = sorted(
    p for p in (dataset / "images/val").iterdir()
    if p.name.startswith(source + "_")
    and p.suffix.lower() in {".jpg", ".jpeg", ".png"}
)

assert len(val_images) == 2401, (
    f"Expected 2401 images, found {len(val_images)}"
)

# Prepare a source-specific image list outside the original dataset
eval_dir = base / "reports/source_validation"
eval_dir.mkdir(parents=True, exist_ok=True)

image_list = eval_dir / "china_drone_images.txt"
image_list.write_text(
    "\n".join(str(p.resolve()) for p in val_images) + "\n"
)

source_yaml = {
    "path": str(dataset),
    "train": "images/train",
    "val": str(image_list),
    "nc": 4,
    "names": {0: "D00", 1: "D10", 2: "D20", 3: "D40"},
}

yaml_path = eval_dir / "china_drone.yaml"
yaml_path.write_text(yaml.safe_dump(source_yaml, sort_keys=False))

print("PASS: China_Drone validation configuration prepared")
print("Images:", len(val_images))
print("Configuration:", yaml_path)


In [ ]:

from pathlib import Path

base = Path("/content/drive/MyDrive/AI Project /M4_Object_Detection")
eval_dir = base / "reports/source_validation"
yaml_path = eval_dir / "china_drone.yaml"

assert yaml_path.is_file(), "China_Drone YAML not found"

china_drone_metrics = model.val(
    data=str(yaml_path),
    imgsz=640,
    batch=16,
    device=0,
    workers=2,
    plots=False,
    project=str(eval_dir),
    name="china_drone_results",
    exist_ok=True
)

print("\nPASS: China_Drone validation completed")
print("Precision:", round(float(china_drone_metrics.box.mp), 4))
print("Recall:", round(float(china_drone_metrics.box.mr), 4))
print("mAP@50:", round(float(china_drone_metrics.box.map50), 4))
print("mAP@50-95:", round(float(china_drone_metrics.box.map), 4))


In [ ]:

from pathlib import Path
import yaml

base = Path("/content/drive/MyDrive/AI Project /M4_Object_Detection")
dataset = base / "data/yolo_dataset"
eval_dir = base / "reports/source_validation"

source = "China_MotorBike"

val_images = sorted(
    p for p in (dataset / "images/val").iterdir()
    if p.name.startswith(source + "_")
    and p.suffix.lower() in {".jpg", ".jpeg", ".png"}
)

assert len(val_images) == 1977, (
    f"Expected 1977 images, found {len(val_images)}"
)

image_list = eval_dir / "china_motorbike_images.txt"
image_list.write_text(
    "\n".join(str(p.resolve()) for p in val_images) + "\n"
)

source_yaml = {
    "path": str(dataset),
    "train": "images/train",
    "val": str(image_list),
    "nc": 4,
    "names": {0: "D00", 1: "D10", 2: "D20", 3: "D40"},
}

yaml_path = eval_dir / "china_motorbike.yaml"
yaml_path.write_text(yaml.safe_dump(source_yaml, sort_keys=False))

print("PASS: China_MotorBike validation configuration prepared")
print("Images:", len(val_images))
print("Configuration:", yaml_path)


In [ ]:

from pathlib import Path

base = Path("/content/drive/MyDrive/AI Project /M4_Object_Detection")
eval_dir = base / "reports/source_validation"
yaml_path = eval_dir / "china_motorbike.yaml"

assert yaml_path.is_file(), "China_MotorBike YAML not found"

china_motorbike_metrics = model.val(
    data=str(yaml_path),
    imgsz=640,
    batch=16,
    device=0,
    workers=2,
    plots=False,
    project=str(eval_dir),
    name="china_motorbike_results",
    exist_ok=True
)

print("\nPASS: China_MotorBike validation completed")
print("Precision:", round(float(china_motorbike_metrics.box.mp), 4))
print("Recall:", round(float(china_motorbike_metrics.box.mr), 4))
print("mAP@50:", round(float(china_motorbike_metrics.box.map50), 4))
print("mAP@50-95:", round(float(china_motorbike_metrics.box.map), 4))


In [ ]:

from pathlib import Path
import yaml

base = Path("/content/drive/MyDrive/AI Project /M4_Object_Detection")
val_images = base / "data/yolo_dataset/images/val"
val_labels = base / "data/yolo_dataset/labels/val"

# Select only Czech validation images
czech_images = sorted(val_images.glob("Czech_*.jpg"))
czech_images += sorted(val_images.glob("Czech_*.png"))
czech_images = sorted(set(czech_images))

assert val_images.exists(), "Validation images folder not found"
assert val_labels.exists(), "Validation labels folder not found"
assert len(czech_images) == 2829, f"Expected 2829, found {len(czech_images)}"

# Create a Czech-only image list without moving any files
report_dir = base / "reports/source_validation"
report_dir.mkdir(parents=True, exist_ok=True)

image_list = report_dir / "czech_images.txt"
image_list.write_text(
    "\n".join(str(p) for p in czech_images) + "\n"
)

# Create source-specific validation configuration
czech_yaml = report_dir / "czech.yaml"

config = {
    "path": str(base / "data/yolo_dataset"),
    "train": "images/train",
    "val": str(image_list),
    "nc": 4,
    "names": {0: "D00", 1: "D10", 2: "D20", 3: "D40"}
}

with open(czech_yaml, "w") as f:
    yaml.safe_dump(config, f, sort_keys=False)

print("PASS: Czech validation configuration prepared")
print("Images:", len(czech_images))
print("Configuration:", czech_yaml)


In [ ]:

from ultralytics import YOLO
from pathlib import Path

base = Path("/content/drive/MyDrive/AI Project /M4_Object_Detection")

checkpoint = base / "runs/yolo11n_baseline/weights/best.pt"
czech_yaml = base / "reports/source_validation/czech.yaml"

assert checkpoint.is_file(), "Best model checkpoint not found"
assert czech_yaml.is_file(), "Czech validation YAML not found"

# Load the trained model; no retraining
model = YOLO(str(checkpoint))

# Evaluate only Czech validation images
results = model.val(
    data=str(czech_yaml),
    imgsz=640,
    batch=16,
    device=0,
    workers=2,
    plots=False,
    project=str(base / "reports/source_validation"),
    name="czech_results",
    exist_ok=True
)

print("\nPASS: Czech validation completed")
print(f"Precision: {results.box.mp:.4f}")
print(f"Recall: {results.box.mr:.4f}")
print(f"mAP@50: {results.box.map50:.4f}")
print(f"mAP@50-95: {results.box.map:.4f}")


In [ ]:

from pathlib import Path
import pandas as pd

base = Path("/content/drive/MyDrive/AI Project /M4_Object_Detection")
images_dir = base / "data/yolo_dataset/images/val"
labels_dir = base / "data/yolo_dataset/labels/val"

sources = {
    "China_Drone": "China_Drone_",
    "China_MotorBike": "China_MotorBike_",
    "Czech": "Czech_"
}

records = []

for source, prefix in sources.items():
    images = [
        p for p in images_dir.iterdir()
        if p.is_file()
        and p.name.startswith(prefix)
        and p.suffix.lower() in {".jpg", ".jpeg", ".png"}
    ]

    annotated = 0
    backgrounds = 0
    total_boxes = 0

    for image in images:
        label_path = labels_dir / (image.stem + ".txt")

        if label_path.exists():
            lines = [
                line for line in label_path.read_text().splitlines()
                if line.strip()
            ]
        else:
            lines = []

        if lines:
            annotated += 1
            total_boxes += len(lines)
        else:
            backgrounds += 1

    records.append({
        "Source": source,
        "Total images": len(images),
        "Annotated images": annotated,
        "Background images": backgrounds,
        "Background %": round(
            100 * backgrounds / len(images), 2
        ) if images else 0,
        "Bounding boxes": total_boxes
    })

summary = pd.DataFrame(records)

display(summary)

assert summary["Total images"].sum() == 7207, \
    "Validation image count mismatch"

print("\nPASS: Validation annotation distribution inspected")


## 12. Prediction visualization and error analysis

Inspect prediction examples, false detections, IoU, confidence thresholds and object-size behavior. Use validation data for diagnostic decisions.

In [ ]:

from pathlib import Path
import random
import matplotlib.pyplot as plt
from PIL import Image

base = Path("/content/drive/MyDrive/AI Project /M4_Object_Detection")
images_dir = base / "data/yolo_dataset/images/val"
labels_dir = base / "data/yolo_dataset/labels/val"

backgrounds = []

for p in sorted(images_dir.iterdir()):
    if not p.name.startswith("Czech_"):
        continue
    if p.suffix.lower() not in {".jpg", ".jpeg", ".png"}:
        continue

    label = labels_dir / f"{p.stem}.txt"
    if not label.exists() or not label.read_text().strip():
        backgrounds.append(p)

assert len(backgrounds) == 1757, (
    f"Expected 1757 Czech backgrounds, found {len(backgrounds)}"
)

rng = random.Random(42)
sample = rng.sample(backgrounds, 12)

fig, axes = plt.subplots(3, 4, figsize=(16, 10))

for ax, path in zip(axes.flat, sample):
    with Image.open(path) as img:
        ax.imshow(img.convert("RGB"))
    ax.set_title(path.name, fontsize=8)
    ax.axis("off")

plt.tight_layout()
plt.show()

print("PASS: Displayed 12 Czech images without annotations")
print("These images are not necessarily damage-free.")


In [ ]:

from pathlib import Path
from ultralytics import YOLO

base = Path("/content/drive/MyDrive/AI Project /M4_Object_Detection")

checkpoint = base / "runs/yolo11n_baseline/weights/best.pt"
czech_yaml = base / "reports/source_validation/czech.yaml"
val_images = base / "data/yolo_dataset/images/val"
val_labels = base / "data/yolo_dataset/labels/val"

checks = {
    "Trained YOLO11n model": checkpoint.is_file(),
    "Czech validation configuration": czech_yaml.is_file(),
    "Validation images": val_images.is_dir(),
    "Validation labels": val_labels.is_dir()
}

for name, passed in checks.items():
    print(f"{name}: {'PASS' if passed else 'NOT FOUND'}")

assert all(checks.values()), "Some required files are missing"

model = YOLO(str(checkpoint))

print("\nPASS: Ready for Czech detection error analysis")


In [ ]:

from pathlib import Path
import random
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from matplotlib.patches import Rectangle

base = Path("/content/drive/MyDrive/AI Project /M4_Object_Detection")
images_dir = base / "data/yolo_dataset/images/val"
labels_dir = base / "data/yolo_dataset/labels/val"

class_names = ["D00", "D10", "D20", "D40"]

# Select Czech images with existing annotations
czech_images = sorted(
    p for p in images_dir.glob("Czech_*")
    if p.suffix.lower() in {".jpg", ".jpeg", ".png"}
    and (labels_dir / f"{p.stem}.txt").is_file()
    and (labels_dir / f"{p.stem}.txt").stat().st_size > 0
)

assert len(czech_images) > 0, "No annotated Czech images found"

rng = random.Random(42)
sample = rng.sample(czech_images, min(6, len(czech_images)))

fig, axes = plt.subplots(2, 3, figsize=(18, 11))

for ax, image_path in zip(axes.flat, sample):
    image = np.array(Image.open(image_path).convert("RGB"))
    h, w = image.shape[:2]

    ax.imshow(image)

    # Ground-truth boxes (green)
    label_path = labels_dir / f"{image_path.stem}.txt"

    for line in label_path.read_text().splitlines():
        values = line.split()
        if len(values) != 5:
            continue

        cls = int(float(values[0]))
        xc, yc, bw, bh = map(float, values[1:])

        x = (xc - bw / 2) * w
        y = (yc - bh / 2) * h

        ax.add_patch(Rectangle(
            (x, y), bw * w, bh * h,
            fill=False, edgecolor="lime", linewidth=2
        ))
        ax.text(x, max(0, y - 5), f"GT {class_names[cls]}",
                color="lime", fontsize=9, weight="bold")

    # YOLO predictions (red)
    prediction = model.predict(
        source=str(image_path),
        imgsz=640,
        conf=0.25,
        device=0,
        verbose=False
    )[0]

    for box in prediction.boxes:
        x1, y1, x2, y2 = box.xyxy[0].cpu().numpy()
        cls = int(box.cls[0])
        confidence = float(box.conf[0])

        ax.add_patch(Rectangle(
            (x1, y1), x2 - x1, y2 - y1,
            fill=False, edgecolor="red", linewidth=2
        ))
        ax.text(x1, min(h - 5, y2 + 12),
                f"Pred {class_names[cls]} {confidence:.2f}",
                color="red", fontsize=9, weight="bold")

    ax.set_title(image_path.name, fontsize=9)
    ax.axis("off")

plt.tight_layout()
plt.show()

print("PASS: Czech ground truth and predictions visualized")
print("Green = Ground truth | Red = Predictions")


In [ ]:

from matplotlib.patches import Rectangle
import matplotlib.pyplot as plt
import numpy as np
from PIL import Image

fig, axes = plt.subplots(2, 3, figsize=(18, 11))
total_predictions = 0

for ax, image_path in zip(axes.flat, sample):
    image = np.array(Image.open(image_path).convert("RGB"))
    ax.imshow(image)

    result = model.predict(
        source=str(image_path),
        imgsz=640,
        conf=0.05,
        device=0,
        verbose=False
    )[0]

    total_predictions += len(result.boxes)

    for box in result.boxes:
        x1, y1, x2, y2 = box.xyxy[0].cpu().numpy()
        cls = int(box.cls[0])
        confidence = float(box.conf[0])

        ax.add_patch(Rectangle(
            (x1, y1), x2-x1, y2-y1,
            fill=False, edgecolor="red", linewidth=2
        ))
        ax.text(
            x1, max(0, y1-5),
            f"{class_names[cls]} {confidence:.2f}",
            color="red", fontsize=9, weight="bold"
        )

    ax.set_title(
        f"{image_path.name} | {len(result.boxes)} predictions",
        fontsize=9
    )
    ax.axis("off")

plt.tight_layout()
plt.show()

print("Total predictions at confidence 0.05:", total_predictions)
print("PASS: Low-confidence prediction inspection completed")


In [ ]:

import numpy as np

def calculate_iou(box1, box2):
    x1 = max(box1[0], box2[0])
    y1 = max(box1[1], box2[1])
    x2 = min(box1[2], box2[2])
    y2 = min(box1[3], box2[3])

    intersection = max(0, x2 - x1) * max(0, y2 - y1)

    area1 = max(0, box1[2] - box1[0]) * max(0, box1[3] - box1[1])
    area2 = max(0, box2[2] - box2[0]) * max(0, box2[3] - box2[1])

    union = area1 + area2 - intersection
    return intersection / union if union > 0 else 0


# Confirm analysis settings
CONF_THRESHOLD = 0.25
IOU_THRESHOLD = 0.50

print("PASS: Error analysis functions prepared")
print("Confidence threshold:", CONF_THRESHOLD)
print("IoU matching threshold:", IOU_THRESHOLD)
print("Dataset: Czech validation only")
print("Model: Existing YOLO11n best.pt")


In [ ]:

from pathlib import Path
import numpy as np
import pandas as pd

base = Path("/content/drive/MyDrive/AI Project /M4_Object_Detection")
images_dir = base / "data/yolo_dataset/images/val"
labels_dir = base / "data/yolo_dataset/labels/val"

czech_images = sorted(
    p for p in images_dir.iterdir()
    if p.name.startswith("Czech_")
    and p.suffix.lower() in {".jpg", ".jpeg", ".png"}
)

assert len(czech_images) == 2829

counts = {
    cls: {"TP": 0, "FP": 0, "FN": 0}
    for cls in range(4)
}

for index, image_path in enumerate(czech_images, 1):

    # Read image dimensions
    from PIL import Image
    with Image.open(image_path) as img:
        width, height = img.size

    # Read ground-truth annotations
    gt = []
    label_path = labels_dir / f"{image_path.stem}.txt"

    if label_path.exists():
        for line in label_path.read_text().splitlines():
            values = line.split()
            if len(values) != 5:
                continue

            cls = int(float(values[0]))
            xc, yc, bw, bh = map(float, values[1:])

            gt.append((
                cls,
                [
                    (xc - bw / 2) * width,
                    (yc - bh / 2) * height,
                    (xc + bw / 2) * width,
                    (yc + bh / 2) * height
                ]
            ))

    # Run predictions using the existing trained model
    result = model.predict(
        source=str(image_path),
        imgsz=640,
        conf=CONF_THRESHOLD,
        device=0,
        verbose=False
    )[0]

    predictions = [
        (
            int(box.cls[0]),
            box.xyxy[0].cpu().numpy().tolist(),
            float(box.conf[0])
        )
        for box in result.boxes
    ]

    # Match predictions to ground truth, highest confidence first
    predictions.sort(key=lambda p: p[2], reverse=True)
    matched_gt = set()

    for pred_cls, pred_box, confidence in predictions:
        best_iou = 0
        best_idx = None

        for gt_idx, (gt_cls, gt_box) in enumerate(gt):
            if gt_idx in matched_gt or gt_cls != pred_cls:
                continue

            iou = calculate_iou(pred_box, gt_box)

            if iou > best_iou:
                best_iou = iou
                best_idx = gt_idx

        if best_idx is not None and best_iou >= IOU_THRESHOLD:
            matched_gt.add(best_idx)
            counts[pred_cls]["TP"] += 1
        else:
            counts[pred_cls]["FP"] += 1

    # Unmatched annotations are false negatives
    for gt_idx, (gt_cls, _) in enumerate(gt):
        if gt_idx not in matched_gt:
            counts[gt_cls]["FN"] += 1

    if index % 500 == 0:
        print(f"Processed {index}/{len(czech_images)} images")

rows = []

for cls, values in counts.items():
    tp, fp, fn = values["TP"], values["FP"], values["FN"]

    rows.append({
        "Class": class_names[cls],
        "TP": tp,
        "FP": fp,
        "FN": fn,
        "Precision": round(tp / (tp + fp), 4) if tp + fp else 0,
        "Recall": round(tp / (tp + fn), 4) if tp + fn else 0
    })

error_summary = pd.DataFrame(rows)
display(error_summary)

totals = error_summary[["TP", "FP", "FN"]].sum()

print("\nOverall detection counts:")
print(totals.to_string())
print("\nPASS: Czech quantitative error analysis completed")


In [ ]:

from pathlib import Path

base = Path("/content/drive/MyDrive/AI Project /M4_Object_Detection")
report_dir = base / "reports/error_analysis"
report_dir.mkdir(parents=True, exist_ok=True)

output_file = report_dir / "czech_error_summary_conf025_iou050.csv"

error_summary.to_csv(output_file, index=False)

assert output_file.is_file(), "Report was not saved"

print("PASS: Czech error-analysis report saved")
print("Location:", output_file)
print("Total TP:", int(error_summary["TP"].sum()))
print("Total FP:", int(error_summary["FP"].sum()))
print("Total FN:", int(error_summary["FN"].sum()))


In [ ]:

import pandas as pd

thresholds = [0.05, 0.10, 0.25]
records = []

for image_path in sample:
    for threshold in thresholds:
        result = model.predict(
            source=str(image_path),
            imgsz=640,
            conf=threshold,
            device=0,
            verbose=False
        )[0]

        records.append({
            "Image": image_path.name,
            "Confidence threshold": threshold,
            "Predictions": len(result.boxes)
        })

comparison = pd.DataFrame(records)

display(
    comparison.pivot(
        index="Image",
        columns="Confidence threshold",
        values="Predictions"
    )
)

print("\nTotal predictions by confidence threshold:")
print(
    comparison.groupby("Confidence threshold")["Predictions"].sum()
)

print("\nPASS: Confidence threshold comparison completed")


In [ ]:

from ultralytics import YOLO
from pathlib import Path

base = Path("/content/drive/MyDrive/AI Project /M4_Object_Detection")

checkpoint = base / "runs/yolo11n_baseline/weights/best.pt"
czech_yaml = base / "reports/source_validation/czech.yaml"

low_conf_model = YOLO(str(checkpoint))

low_conf_results = low_conf_model.val(
    data=str(czech_yaml),
    imgsz=640,
    batch=16,
    conf=0.05,
    iou=0.7,
    device=0,
    workers=2,
    plots=False,
    project=str(base / "reports/error_analysis"),
    name="czech_conf005",
    exist_ok=True
)

print("\nPASS: Czech low-confidence evaluation completed")
print(f"Precision: {low_conf_results.box.mp:.4f}")
print(f"Recall: {low_conf_results.box.mr:.4f}")
print(f"mAP50: {low_conf_results.box.map50:.4f}")
print(f"mAP50-95: {low_conf_results.box.map:.4f}")


In [ ]:

from pathlib import Path
import pandas as pd

base = Path("/content/drive/MyDrive/AI Project /M4_Object_Detection")
report_dir = base / "reports/error_analysis"
report_dir.mkdir(parents=True, exist_ok=True)

comparison = pd.DataFrame([
    {
        "Confidence": 0.25,
        "Precision": 0.2297,
        "Recall": 0.2054,
        "mAP50": 0.1427,
        "mAP50_95": 0.0480
    },
    {
        "Confidence": 0.05,
        "Precision": float(low_conf_results.box.mp),
        "Recall": float(low_conf_results.box.mr),
        "mAP50": float(low_conf_results.box.map50),
        "mAP50_95": float(low_conf_results.box.map)
    }
])

output = report_dir / "czech_confidence_comparison.csv"
comparison.to_csv(output, index=False)

display(comparison)

print("\nPASS: Czech confidence comparison saved")
print("Location:", output)


In [ ]:

from pathlib import Path
import pandas as pd

base = Path("/content/drive/MyDrive/AI Project /M4_Object_Detection")
labels_dir = base / "data/yolo_dataset/labels/val"

records = []

for label_path in sorted(labels_dir.glob("Czech_*.txt")):
    for line in label_path.read_text().splitlines():
        values = line.split()

        if len(values) != 5:
            continue

        cls = int(float(values[0]))
        xc, yc, width, height = map(float, values[1:])

        # YOLO normalized bounding-box area
        area_percent = width * height * 100

        records.append({
            "Class": ["D00", "D10", "D20", "D40"][cls],
            "Area %": area_percent,
            "Width %": width * 100,
            "Height %": height * 100
        })

box_sizes = pd.DataFrame(records)

assert len(box_sizes) == 1745, (
    f"Expected 1745 boxes, found {len(box_sizes)}"
)

# Descriptive categories for this analysis only
box_sizes["Size category"] = pd.cut(
    box_sizes["Area %"],
    bins=[-0.001, 0.1, 1.0, 100.0],
    labels=["Tiny (<0.1%)", "Small (0.1–1%)", "Larger (>1%)"]
)

summary = pd.crosstab(
    box_sizes["Class"],
    box_sizes["Size category"]
)

display(summary)

print("\nMedian bounding-box area by class (% of image):")
print(box_sizes.groupby("Class")["Area %"].median().round(4))

print("\nTotal annotations:", len(box_sizes))
print("PASS: Czech object-size analysis completed")


In [ ]:

from pathlib import Path

base = Path("/content/drive/MyDrive/AI Project /M4_Object_Detection")
report_dir = base / "reports/error_analysis"
report_dir.mkdir(parents=True, exist_ok=True)

output = report_dir / "czech_object_size_analysis.csv"

box_sizes.to_csv(output, index=False)

assert output.exists()

print("PASS: Czech object-size analysis saved")
print("Location:", output)
print("Total annotations:", len(box_sizes))


## 13. Czech validation diagnostics

Preserve the Czech validation reports and object-size/error-analysis workflow.

In [ ]:

from pathlib import Path

base = Path("/content/drive/MyDrive/AI Project /M4_Object_Detection")
report_dir = base / "reports/error_analysis"

required_files = {
    "Czech error analysis": "czech_error_summary_conf025_iou050.csv",
    "Confidence comparison": "czech_confidence_comparison.csv",
    "Object-size analysis": "czech_object_size_analysis.csv",
    "Trained YOLO11n model": "../../runs/yolo11n_baseline/weights/best.pt",
}

for name, relative_path in required_files.items():
    path = report_dir / relative_path
    status = "PASS" if path.is_file() else "NOT FOUND"
    print(f"{name}: {status}")

print("\nVerification completed.")


In [ ]:

from pathlib import Path
import pandas as pd
from ultralytics import YOLO

base = Path("/content/drive/MyDrive/AI Project /M4_Object_Detection")
report_dir = base / "reports/error_analysis"

# Restore previously saved analysis
error_summary = pd.read_csv(
    report_dir / "czech_error_summary_conf025_iou050.csv"
)

box_sizes = pd.read_csv(
    report_dir / "czech_object_size_analysis.csv"
)

# Restore trained YOLO11n model
checkpoint = base / "runs/yolo11n_baseline/weights/best.pt"
model = YOLO(str(checkpoint))

print("PASS: Czech error analysis restored")
print("PASS: Object-size data restored")
print("PASS: Trained YOLO11n model loaded")

print("\nPrevious error summary:")
display(error_summary)

print("\nTotal annotated objects:", len(box_sizes))
print("Model classes:", model.names)


In [ ]:

from pathlib import Path
import pandas as pd
from ultralytics import YOLO

base = Path("/content/drive/MyDrive/AI Project /M4_Object_Detection")
report_dir = base / "reports/error_analysis"

# Restore existing reports
error_summary = pd.read_csv(
    report_dir / "czech_error_summary_conf025_iou050.csv"
)

box_sizes = pd.read_csv(
    report_dir / "czech_object_size_analysis.csv"
)

# Load previously trained model
checkpoint = base / "runs/yolo11n_baseline/weights/best.pt"
model = YOLO(str(checkpoint))

print("PASS: Czech error analysis restored")
print("PASS: Object-size analysis restored")
print("PASS: YOLO11n best checkpoint loaded")

print("\nCzech error summary:")
display(error_summary)

print("\nTotal annotations:", len(box_sizes))
print("Model classes:", model.names)


In [ ]:

from pathlib import Path

base = Path("/content/drive/MyDrive/AI Project /M4_Object_Detection")
images_dir = base / "data/yolo_dataset/images/val"
labels_dir = base / "data/yolo_dataset/labels/val"

czech_images = sorted(
    p for p in images_dir.iterdir()
    if p.name.startswith("Czech_")
    and p.suffix.lower() in {".jpg", ".jpeg", ".png"}
)

def calculate_iou(box1, box2):
    x1 = max(box1[0], box2[0])
    y1 = max(box1[1], box2[1])
    x2 = min(box1[2], box2[2])
    y2 = min(box1[3], box2[3])

    intersection = max(0, x2-x1) * max(0, y2-y1)

    area1 = max(0, box1[2]-box1[0]) * max(0, box1[3]-box1[1])
    area2 = max(0, box2[2]-box2[0]) * max(0, box2[3]-box2[1])

    union = area1 + area2 - intersection

    return intersection / union if union > 0 else 0

CONF_THRESHOLD = 0.25
IOU_THRESHOLD = 0.50

assert len(czech_images) == 2829, "Unexpected Czech image count"
assert images_dir.exists() and labels_dir.exists()

print("PASS: Czech validation images:", len(czech_images))
print("PASS: Matching function prepared")
print("Confidence threshold:", CONF_THRESHOLD)
print("IoU threshold:", IOU_THRESHOLD)
print("PASS: Ready for object-size detection analysis")


In [ ]:

import pandas as pd
from PIL import Image

records = []

for index, image_path in enumerate(czech_images, 1):

    with Image.open(image_path) as img:
        width, height = img.size

    label_path = labels_dir / f"{image_path.stem}.txt"
    ground_truth = []

    if label_path.exists():
        for line in label_path.read_text().splitlines():
            values = line.split()
            if len(values) != 5:
                continue

            cls = int(float(values[0]))
            xc, yc, bw, bh = map(float, values[1:])

            ground_truth.append({
                "class_id": cls,
                "box": [
                    (xc - bw/2) * width,
                    (yc - bh/2) * height,
                    (xc + bw/2) * width,
                    (yc + bh/2) * height
                ],
                "area_percent": bw * bh * 100
            })

    result = model.predict(
        source=str(image_path),
        imgsz=640,
        conf=CONF_THRESHOLD,
        device=0,
        verbose=False
    )[0]

    predictions = sorted(
        [
            (
                int(box.cls[0]),
                box.xyxy[0].cpu().numpy().tolist(),
                float(box.conf[0])
            )
            for box in result.boxes
        ],
        key=lambda x: x[2],
        reverse=True
    )

    matched = set()

    for pred_cls, pred_box, confidence in predictions:
        best_iou = 0
        best_idx = None

        for gt_idx, gt in enumerate(ground_truth):
            if gt_idx in matched or gt["class_id"] != pred_cls:
                continue

            iou = calculate_iou(pred_box, gt["box"])

            if iou > best_iou:
                best_iou = iou
                best_idx = gt_idx

        if best_idx is not None and best_iou >= IOU_THRESHOLD:
            matched.add(best_idx)

    for gt_idx, gt in enumerate(ground_truth):
        area = gt["area_percent"]

        size = (
            "Tiny (<0.1%)" if area < 0.1
            else "Small (0.1–1%)" if area <= 1.0
            else "Larger (>1%)"
        )

        records.append({
            "Image": image_path.name,
            "Class": model.names[gt["class_id"]],
            "Area %": area,
            "Size": size,
            "Detected": int(gt_idx in matched)
        })

    if index % 500 == 0:
        print(f"Processed {index}/{len(czech_images)} images")

size_detection = pd.DataFrame(records)

assert len(size_detection) == 1745, (
    f"Expected 1745 annotations, found {len(size_detection)}"
)

summary = (
    size_detection.groupby(["Class", "Size"])["Detected"]
    .agg(Annotations="count", Detected="sum", Recall="mean")
    .reset_index()
)

summary["Recall"] = (summary["Recall"] * 100).round(2)

display(summary)

# Verify consistency with previous error analysis
assert size_detection["Detected"].sum() == 120, (
    "Matched detections differ from previous Czech analysis"
)

# Save directly to Google Drive
report_dir = base / "reports/error_analysis"
report_dir.mkdir(parents=True, exist_ok=True)

size_detection.to_csv(
    report_dir / "czech_detection_by_object_size.csv",
    index=False
)

summary.to_csv(
    report_dir / "czech_size_recall_summary.csv",
    index=False
)

print("\nPASS: Object-size detection analysis completed")
print("Total annotations:", len(size_detection))
print("Matched detections:", size_detection["Detected"].sum())
print("Reports saved to Google Drive")


In [ ]:

from pathlib import Path

base = Path("/content/drive/MyDrive/AI Project /M4_Object_Detection")

folders = [
    base / "runs/yolo11n_baseline",
    base / "reports/error_analysis",
    base / "reports/source_validation"
]

for folder in folders:
    print(f"\nFolder: {folder.name}")

    if folder.exists():
        for file in sorted(folder.iterdir()):
            print(" -", file.name)
    else:
        print(" NOT FOUND")

print("\nPASS: Existing diagnostic artifacts inspected")


In [ ]:

from pathlib import Path
from PIL import Image
import matplotlib.pyplot as plt

base = Path("/content/drive/MyDrive/AI Project /M4_Object_Detection")
run_dir = base / "runs/yolo11n_baseline"

matrix_files = [
    run_dir / "confusion_matrix.png",
    run_dir / "confusion_matrix_normalized.png"
]

fig, axes = plt.subplots(1, 2, figsize=(18, 8))

for ax, path in zip(axes, matrix_files):
    assert path.is_file(), f"Missing file: {path.name}"

    image = Image.open(path)
    ax.imshow(image)
    ax.set_title(path.stem.replace("_", " ").title())
    ax.axis("off")

plt.tight_layout()
plt.show()

print("PASS: Existing YOLO11n confusion matrices displayed")
print("Dataset: Combined validation sources")
print("No additional model evaluation performed")


In [ ]:

from pathlib import Path
import yaml

run_dir = Path(
    "/content/drive/MyDrive/AI Project /M4_Object_Detection/runs/yolo11n_baseline"
)

with open(run_dir / "args.yaml", "r") as f:
    args = yaml.safe_load(f)

for key in [
    "model", "data", "epochs", "imgsz", "batch",
    "conf", "iou", "plots", "seed", "deterministic"
]:
    print(f"{key}: {args.get(key, 'Not explicitly specified')}")

print("\nPASS: Baseline configuration inspected")


In [ ]:

from pathlib import Path

base = Path("/content/drive/MyDrive/AI Project /M4_Object_Detection")
source_dir = base / "reports/source_validation"

for source in ["china_drone", "china_motorbike", "czech"]:
    print(f"\n{source.upper()}")

    matches = sorted(source_dir.glob(f"{source}*"))

    if not matches:
        print("No saved artifacts found")
        continue

    for path in matches:
        print(
            "DIR " if path.is_dir() else "FILE",
            path.name
        )

        if path.is_dir():
            for item in sorted(path.iterdir()):
                print("   -", item.name)

print("\nPASS: Saved source-validation artifacts inspected")


In [ ]:

from pathlib import Path
import pandas as pd

base = Path("/content/drive/MyDrive/AI Project /M4_Object_Detection")
report_dir = base / "reports/source_validation"

validation_summary = pd.DataFrame([
    {
        "Source": "China_Drone",
        "Images": 2401,
        "Precision": 0.2987,
        "Recall": 0.2953,
        "mAP50": 0.2130,
        "mAP50_95": 0.0901
    },
    {
        "Source": "China_MotorBike",
        "Images": 1977,
        "Precision": 0.3588,
        "Recall": 0.3148,
        "mAP50": 0.2603,
        "mAP50_95": 0.1078
    },
    {
        "Source": "Czech",
        "Images": 2829,
        "Precision": 0.2297,
        "Recall": 0.2054,
        "mAP50": 0.1427,
        "mAP50_95": 0.0480
    }
])

assert validation_summary["Images"].sum() == 7207

output = report_dir / "m4_validation_source_comparison.csv"
validation_summary.to_csv(output, index=False)

display(validation_summary)

print("\nPASS: Validation comparison saved")
print("Location:", output)
print("Total validation images:", validation_summary["Images"].sum())


## 14. Independent Norway test preparation

Norway is held out from training and validation. Restoration of Norway test images is a one-time data-recovery operation.

In [ ]:

import pandas as pd
import zipfile
from pathlib import Path

base = Path("/content/drive/MyDrive/AI Project ")

manifest = pd.read_csv(base / "Data/split_manifest.csv")

norway = manifest[
    manifest["split"].astype(str).str.lower().eq("test")
].copy()

zip_path = (
    base / "Data/RDD2022_released_through_CRDDC2022/"
    "RDD2022/Norway.zip"
)

with zipfile.ZipFile(zip_path) as z:
    zip_names = {
        Path(name).name
        for name in z.namelist()
        if name.lower().endswith((".jpg", ".jpeg", ".png"))
    }

norway["original_filename"] = (
    norway["image_id"]
    .astype(str)
    .str.replace(r"^Norway-002_", "", regex=True)
    + ".jpg"
)

norway["found_in_zip"] = norway["original_filename"].isin(zip_names)

print("M1 Norway test images:", len(norway))
print("Matching ZIP images:", norway["found_in_zip"].sum())
print("Missing ZIP images:", (~norway["found_in_zip"]).sum())

print("\nSample mappings:")
display(
    norway[
        ["image_id", "original_filename", "found_in_zip"]
    ].head(10)
)

print("\nUnique M1 image IDs:", norway["image_id"].nunique())
print("No files extracted or modified")


In [ ]:

from pathlib import Path
import zipfile
import shutil
import pandas as pd

base = Path("/content/drive/MyDrive/AI Project ")

zip_path = (
    base / "Data/RDD2022_released_through_CRDDC2022/"
    "RDD2022/Norway.zip"
)

manifest = pd.read_csv(base / "Data/split_manifest.csv")

norway = manifest[
    manifest["split"].astype(str).str.lower().eq("test")
].copy()

assert len(norway) == 8161
assert norway["image_id"].nunique() == 8161
assert norway["image_id"].str.startswith("Norway-002_").all()

destination = (
    base / "M4_Object_Detection/data/yolo_dataset/images/test"
)
destination.mkdir(parents=True, exist_ok=True)

restored = 0
already_present = 0

with zipfile.ZipFile(zip_path) as z:
    archive_lookup = {
        Path(name).name: name
        for name in z.namelist()
        if name.lower().endswith(".jpg")
    }

    expected = {
        image_id: image_id.removeprefix("Norway-002_") + ".jpg"
        for image_id in norway["image_id"]
    }

    assert all(
        filename in archive_lookup
        for filename in expected.values()
    ), "Some Norway images are missing from the archive"

    for image_id, original_name in expected.items():
        output_file = destination / f"{image_id}.jpg"

        if output_file.exists():
            already_present += 1
            continue

        with z.open(archive_lookup[original_name]) as source:
            with open(output_file, "wb") as target:
                shutil.copyfileobj(source, target)

        restored += 1

        if restored % 500 == 0:
            print(f"Restored {restored}/8161 images")

print("\nPASS: Norway restoration completed")
print("Images restored:", restored)
print("Already present:", already_present)
print("Destination:", destination)


In [ ]:

from pathlib import Path
from PIL import Image
import pandas as pd
import random

base = Path("/content/drive/MyDrive/AI Project ")
dataset = base / "M4_Object_Detection/data/yolo_dataset"

manifest = pd.read_csv(base / "Data/split_manifest.csv")
test = manifest[manifest["split"].astype(str).str.lower().eq("test")]

image_dir = dataset / "images/test"
label_dir = dataset / "labels/test"

expected = {f"{image_id}.jpg" for image_id in test["image_id"]}
actual = {p.name for p in image_dir.glob("*.jpg")}

print("Expected Norway images:", len(expected))
print("Actual Norway images:", len(actual))
print("Missing images:", len(expected - actual))
print("Unexpected images:", len(actual - expected))
print("Label files:", len(list(label_dir.glob("*.txt"))))

assert expected == actual, "Test image filenames do not match M1"
assert len(list(label_dir.glob("*.txt"))) == 2914

random.seed(42)
samples = random.sample(sorted(expected), 20)

for filename in samples:
    with Image.open(image_dir / filename) as img:
        img.verify()

print("Random image integrity checks: 20/20 passed")
print("\nPASS: Norway test dataset verified")
print("No inference or training performed")


## 15. Final Norway evaluation

Final evaluation uses the saved best YOLO11n checkpoint on 8,161 Norway images. Do not tune the model based on these results.

In [ ]:

from pathlib import Path
import yaml

base = Path("/content/drive/MyDrive/AI Project /M4_Object_Detection")
dataset = base / "data/yolo_dataset"

model_path = base / "runs/yolo11n_baseline/weights/best.pt"
test_images = dataset / "images/test"
test_labels = dataset / "labels/test"

assert model_path.is_file(), "Trained model not found"
assert test_images.is_dir(), "Norway images not found"
assert test_labels.is_dir(), "Norway labels not found"

test_config = {
    "path": str(dataset),
    "train": "images/train",
    "val": "images/test",
    "test": "images/test",
    "nc": 4,
    "names": {
        0: "D00",
        1: "D10",
        2: "D20",
        3: "D40"
    }
}

config_path = base / "data/norway_test.yaml"

with open(config_path, "w") as f:
    yaml.safe_dump(test_config, f, sort_keys=False)

print("PASS: Norway test configuration prepared")
print("Model:", model_path)
print("Configuration:", config_path)
print("Test images:", test_images)
print("No evaluation or training performed")


In [ ]:

from ultralytics import YOLO
from pathlib import Path

base = Path("/content/drive/MyDrive/AI Project /M4_Object_Detection")

model_path = base / "runs/yolo11n_baseline/weights/best.pt"
config_path = base / "data/norway_test.yaml"

assert model_path.is_file(), "Trained model not found"
assert config_path.is_file(), "Norway configuration not found"

# Load the existing trained model; no retraining
norway_model = YOLO(str(model_path))

# Final independent holdout evaluation
norway_results = norway_model.val(
    data=str(config_path),
    split="test",
    imgsz=640,
    batch=16,
    device=0,
    workers=2,
    conf=0.001,
    iou=0.7,
    plots=True,
    project=str(base / "runs"),
    name="norway_final_test",
    exist_ok=False
)

print("\nPASS: Final Norway evaluation completed")
print("Precision:", round(norway_results.box.mp, 4))
print("Recall:", round(norway_results.box.mr, 4))
print("mAP@50:", round(norway_results.box.map50, 4))
print("mAP@50-95:", round(norway_results.box.map, 4))
print("Results directory:", norway_results.save_dir)


## 16. Reporting and reproducibility

Review saved final metrics, deliverables, software configuration, and training parameters.

In [ ]:

from pathlib import Path
import pandas as pd

base = Path("/content/drive/MyDrive/AI Project /M4_Object_Detection")
report_dir = base / "reports"
report_dir.mkdir(parents=True, exist_ok=True)

summary = pd.DataFrame([{
    "Model": "YOLO11n",
    "Dataset": "Norway-002",
    "Split": "Independent test",
    "Images": 8161,
    "Instances": 11229,
    "Precision": norway_results.box.mp,
    "Recall": norway_results.box.mr,
    "mAP50": norway_results.box.map50,
    "mAP50_95": norway_results.box.map,
    "Image_size": 640,
    "Confidence": 0.001,
    "NMS_IoU": 0.7,
    "Training_epochs": 30,
    "Seed": 42
}])

output = report_dir / "m4_norway_final_test_summary.csv"
summary.to_csv(output, index=False)

display(summary)
print("\nPASS: Final Norway results saved")
print("Location:", output)


In [ ]:
from pathlib import Path

base = Path("/content/drive/MyDrive/AI Project /M4_Object_Detection")

required_files = {
    "Trained YOLO11n model": base / "runs/yolo11n_baseline/weights/best.pt",
    "Training history": base / "runs/yolo11n_baseline/results.csv",
    "Validation source comparison": base / "reports/source_validation/m4_validation_source_comparison.csv",
    "Czech error analysis": base / "reports/error_analysis/czech_error_summary_conf025_iou050.csv",
    "Czech object-size analysis": base / "reports/error_analysis/czech_size_recall_summary.csv",
    "Final Norway test summary": base / "reports/m4_norway_final_test_summary.csv",
    "Norway test results": base / "runs/norway_final_test",
}

print("M4 DELIVERABLE VERIFICATION\n")

missing = []

for name, path in required_files.items():
    exists = path.exists()
    print(f"{'PASS' if exists else 'MISSING'}: {name}")
    if not exists:
        missing.append(name)

print("\nVerification completed.")
print(f"Available: {len(required_files) - len(missing)}/{len(required_files)}")

if missing:
    print("Items requiring attention:", missing)
else:
    print("PASS: All checked M4 deliverables are available.")

In [ ]:
from pathlib import Path
import yaml
import pandas as pd

base = Path("/content/drive/MyDrive/AI Project /M4_Object_Detection")
run_dir = base / "runs/yolo11n_baseline"

config_path = run_dir / "args.yaml"
history_path = run_dir / "results.csv"
checkpoint_path = run_dir / "weights/best.pt"

print("M4 REPRODUCIBILITY CHECK\n")

if config_path.exists():
    with open(config_path, "r") as f:
        config = yaml.safe_load(f)

    for key in [
        "model", "data", "epochs", "imgsz", "batch",
        "seed", "deterministic", "patience", "optimizer"
    ]:
        print(f"{key}: {config.get(key, 'Not recorded')}")
else:
    print("WARNING: Training configuration not found")

if history_path.exists():
    history = pd.read_csv(history_path)
    print("\nRecorded training epochs:", len(history))
    print("Final recorded epoch:", history["epoch"].iloc[-1])
else:
    print("WARNING: Training history not found")

print("\nBest checkpoint exists:", checkpoint_path.is_file())
print("\nPASS: Read-only reproducibility inspection completed")

## 17. Final M4 Evaluation and Failure Analysis

This section consolidates the saved YOLO11n evaluation results without
retraining or modifying the original dataset splits.

- **Validation:** 6,687 annotated images across China Drone, China MotorBike, and Czech.
- **Excluded:** 520 images absent from the validation manifest and without M1 annotation records.
- **Independent test:** Norway (8,161 images), reserved for final generalization assessment.
- **Metrics:** AP50 and AP50–95 by damage class and source.
- **Failure analysis:** Existing validation prediction figures and Czech object-size recall.

The source-level AP evaluations use the existing `best.pt` checkpoint,
640-pixel input size, confidence 0.001, and NMS IoU 0.7.

Norway test results are not used for model selection or threshold tuning.


In [ ]:
from pathlib import Path
import pandas as pd

base = Path("/content/drive/MyDrive/AI Project /M4_Object_Detection")

ap_file = base / "reports/m4_per_class_per_source_ap.csv"
ap = pd.read_csv(ap_file)

print("AP per damage class and source")
display(ap)

print("\nAP50 comparison")
display(ap.pivot(index="Class", columns="Source", values="AP50"))

print("\nAP50–95 comparison")
display(ap.pivot(index="Class", columns="Source", values="AP50_95"))


### 17.1 Failure cases and localization difficulty

The saved baseline validation prediction figures provide qualitative
examples of missed detections and localization challenges.

The Czech object-size analysis shows that D20 recall decreased from
19.73% for larger bounding boxes to 0% for small bounding boxes.
For D40, small-object recall was 3.61%, and none of the three tiny
annotations were detected.

These findings indicate detection difficulty for some small road defects.
However, bounding-box area does not directly measure crack width, and
the observed size effect is not uniform across all classes.

Thin and elongated cracks can also be difficult to represent accurately
with rectangular bounding boxes. Future work could investigate
higher-resolution detection or segmentation.


In [ ]:
from pathlib import Path
from PIL import Image
import pandas as pd
import matplotlib.pyplot as plt

base = Path("/content/drive/MyDrive/AI Project /M4_Object_Detection")

size_report = pd.read_csv(
    base / "reports/error_analysis/czech_size_recall_summary.csv"
)

display(size_report)

folder = base / "reports/failure_cases"

fig, axes = plt.subplots(1, 2, figsize=(16, 8))

for ax, filename, title in zip(
    axes,
    ["czech_failure_ground_truth.jpg", "czech_failure_predictions.jpg"],
    ["Validation ground truth", "Validation predictions"]
):
    ax.imshow(Image.open(folder / filename))
    ax.set_title(title)
    ax.axis("off")

plt.tight_layout()
plt.show()
